# Chapter 8 総合演習：LangChainエコシステム演習問題5問
このノートブックでは、LangChainエコシステムを活用した実践的な演習問題に取り組みます。各演習では、実際のビジネスシナリオや開発現場で直面する課題を解決するためのコードを実装します。

## 必要なライブラリのインストール

まずは必要なライブラリをインストールします。Google Colaboratory環境で実行するための準備です。

In [ ]:
!pip install -U \
  "requests==2.32.5" \
  "langchain==0.3.27" \
  "langchain-core==0.3.79" \
  "langchain-community==0.3.31" \
  "langsmith>=0.3.45,<1.0.0" \
  "langchain-google-genai>=2.1.10,<3.0.0"


## Google API キーの設定

Google Gemini APIを使用するためのAPIキーを設定します。Colabのシークレットから取得します。

In [ ]:
# Google API キーの設定
import os
from dotenv import load_dotenv
load_dotenv()

# Colabのシークレットから API キーを取得
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

# APIキーが正しく設定されたか確認（最初の数文字だけ表示）
api_key = os.environ.get("GOOGLE_API_KEY", "")
if api_key:
    print(f"APIキーが設定されました: {api_key[:5]}...")
else:
    print("APIキーが設定されていません。Colabのシークレットに'GOOGLE_API_KEY'を追加してください。")

APIキーが設定されました: AIzaS...


## 演習1：インテリジェントカスタマーサポートボット
　あなたは、Eコマースサイトの顧客サポート部門のエンジニアです。顧客から
の問い合わせを効率的に処理するために、以下の機能を持つインテリジェントな
カスタマーサポートボットを開発してください。
- 商品情報（提供されたJSON）にアクセスして質問に答える
- 顧客の過去の対話履歴を覚えておく
- 商品の返品プロセスや配送状況など、一般的な質問に回答する
- 複雑な問い合わせは人間のエージェントにエスカレーションする

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_core.runnables import RunnablePassthrough
from langchain.chains import LLMChain

# 商品データ
product_data = {
    "プレミアムヘッドフォン": {"price": 25000, "stock": 10, "warranty": "2年", "return_period": "30日間"},
    "ワイヤレスイヤホン": {"price": 15000, "stock": 5, "warranty": "1年", "return_period": "14日間"},
    "Bluetoothスピーカー": {"price": 8000, "stock": 0, "warranty": "1年", "return_period": "14日間"}
}

# 商品情報を検索する関数
def get_product_info(product_name):
    """商品名から情報を検索する"""
    # 完全一致
    if product_name in product_data:
        info = product_data[product_name]
        return f"商品: {product_name}, 価格: {info['price']}円, 在庫: {info['stock']}個, 保証期間: {info['warranty']}, 返品期間: {info['return_period']}"

    # 部分一致
    for name, info in product_data.items():
        if product_name.lower() in name.lower():
            return f"商品: {name}, 価格: {info['price']}円, 在庫: {info['stock']}個, 保証期間: {info['warranty']}, 返品期間: {info['return_period']}"

    return "商品が見つかりませんでした"

# 返品ポリシー
return_policy = """
返品ポリシー:
1. 未開封・未使用の商品は購入後30日以内であれば全額返金可能です。
2. 開封後の商品は、不良品の場合のみ交換または返金可能です。
3. 返品時の送料は、不良品の場合は当社負担、お客様都合の場合はお客様負担となります。
"""

# 配送データ
shipping_db = {
    "ORD12345": "配送中 - 3月30日到着予定",
    "ORD67890": "発送準備中 - 3月31日発送予定",
    "ORD54321": "配達完了 - 3月25日"
}

# Google API キーの設定
api_key = os.environ.get("GOOGLE_API_KEY", "")
if not api_key:
    raise ValueError("GOOGLE_API_KEY environment variable is not set")

# LLMの初期化
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.2
)

# 顧客クエリを分析する関数
def analyze_query(query):
    """顧客の質問を分析して適切な情報を取得する"""
    # 商品検索
    for product_name in product_data.keys():
        if product_name in query:
            return {"type": "product", "info": get_product_info(product_name)}

    # キーワードベースの検索
    if "ヘッドフォン" in query:
        return {"type": "product", "info": get_product_info("プレミアムヘッドフォン")}
    if "イヤホン" in query:
        return {"type": "product", "info": get_product_info("ワイヤレスイヤホン")}
    if "スピーカー" in query:
        return {"type": "product", "info": get_product_info("Bluetoothスピーカー")}

    # 返品ポリシー
    if "返品" in query or "返金" in query or "ポリシー" in query:
        return {"type": "policy", "info": return_policy}

    # 配送状況
    for order_id in shipping_db.keys():
        if order_id in query:
            return {"type": "shipping", "info": f"注文 {order_id}: {shipping_db[order_id]}"}

    # 該当なし
    return {"type": "unknown", "info": "該当する情報が見つかりませんでした。具体的な商品名や注文番号を教えていただけますか？"}

# プロンプトの定義
template = ChatPromptTemplate.from_messages([
    ("system", """あなたはEコマースサイトの顧客サポートアシスタントです。丁寧で親切な対応を心がけてください。

参考情報:
{context}

この情報を参考に、顧客の質問に分かりやすく回答してください。情報がない場合は、より詳細を聞くか、人間のオペレーターに転送することを提案してください。
"""),
    ("human", "{query}"),
])

# メモリの設定
memory = ConversationBufferMemory(
    input_key="query",
    memory_key="chat_history"
)

# チェーンの定義
def customer_support_chain():
    def get_context(query):
        # 顧客の質問を分析して必要な情報を取得
        result = analyze_query(query)
        return result["info"]

    chain = (
        {"query": RunnablePassthrough(), "context": get_context}
        | template
        | llm
    )

    return chain

# 実行関数
def run_customer_support(query):
    """顧客サポートチェーンを実行"""
    try:
        chain = customer_support_chain()
        response = chain.invoke(query)
        return response.content
    except Exception as e:
        return f"エラーが発生しました: {str(e)}"

/tmp/ipython-input-498705401.py:98: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
# テスト実行
print(run_customer_support("プレミアムヘッドフォンの価格と在庫を教えてください"))

お問い合わせありがとうございます。プレミアムヘッドフォンの価格は25,000円で、現在の在庫は10個となっております。ご検討いただければ幸いです。何か他に気になる点はございますか？


In [ ]:
print(run_customer_support("返品ポリシーについて教えてください"))

はい、返品ポリシーについてご説明いたします。

当サイトでは、お客様に安心して商品をご購入いただけるよう、以下の返品ポリシーを設けております。

*   **未開封・未使用の商品:** 購入後30日以内であれば、全額返金いたします。
*   **開封後の商品:** 不良品の場合のみ、交換または返金が可能です。

また、返品時の送料については、以下の通りです。

*   **不良品の場合:** 当社が負担いたします。
*   **お客様都合の場合:** お客様にご負担いただきます。

ご不明な点がございましたら、お気軽にお問い合わせください。


In [ ]:
print(run_customer_support("注文番号ORD12345の配送状況を確認したいです"))

お問い合わせありがとうございます。ご注文番号ORD12345の配送状況は、現在「配送中」となっており、3月30日にお手元に到着予定です。

何かご不明な点がございましたら、お気軽にお尋ねください。


In [ ]:
print(run_customer_support("前に質問したヘッドフォンの保証期間はどのくらいですか？"))

お問い合わせありがとうございます。以前お問い合わせいただいたプレミアムヘッドフォンの保証期間は2年間でございます。ご安心してお使いいただけるかと思います。他に何かご質問はございますか？


## 演習2：ドキュメント分析エージェント

　あなたは、法律事務所のテクノロジー部門で働いています。弁護士が大量の契
約書を効率的に分析できるよう、以下の機能を持つドキュメント分析エージェン
トを開発してください。
- PDFやWord形式の契約書を読み込む
- 重要な条項（期間、金額、解約条件など）を自動的に抽出する
- 複数の契約書を比較し、相違点を報告する
- 潜在的なリスクや不明確な条項を指摘する

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
# 修正: インポートパスを変更
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.chains import create_extraction_chain
from langchain.chains import SequentialChain, LLMChain
from pydantic import BaseModel, Field
from typing import Optional, List

# 現代的なアプローチ用のPydanticモデル定義
class ContractAnalysis(BaseModel):
    """契約書の重要条項分析結果"""
    contract_period: Optional[str] = Field(None, description="契約の開始日と終了日")
    payment_terms: Optional[str] = Field(None, description="支払い条件、金額、支払いスケジュール")
    termination_clause: Optional[str] = Field(None, description="契約解除/解約の条件")
    confidentiality: Optional[str] = Field(None, description="機密保持条項の詳細")
    limitation_of_liability: Optional[str] = Field(None, description="責任の制限に関する条項")
    governing_law: Optional[str] = Field(None, description="準拠法と管轄裁判所")

class RiskAnalysis(BaseModel):
    """リスク分析結果"""
    unclear_terms: Optional[str] = Field(None, description="不明確または曖昧な表現")
    favorable_conditions: Optional[str] = Field(None, description="一方に有利な条件")
    industry_deviations: Optional[str] = Field(None, description="業界標準と比較した際の特異点")
    legal_risks: Optional[str] = Field(None, description="法的リスク")

class ComparisonResult(BaseModel):
    """契約書比較結果"""
    content_differences: Optional[str] = Field(None, description="各条項の内容の違い")
    favorable_analysis: Optional[str] = Field(None, description="どちらの契約書がより有利な条件か")
    missing_clauses: Optional[str] = Field(None, description="重要な条項の有無")

# ドキュメントローダーの設定
def load_document(file_path):
    """PDFまたはDocxファイルを読み込む"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"ファイルが見つかりません: {file_path}")

    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
        return loader.load()
    elif file_path.endswith('.docx'):
        loader = Docx2txtLoader(file_path)
        return loader.load()
    else:
        raise ValueError("サポートされていないファイル形式です。PDFまたはDocxを使用してください。")

# テキスト分割
def split_documents(documents):
    """ドキュメントをチャンクに分割"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    return text_splitter.split_documents(documents)

# ベクトルストアの作成（エラーハンドリング付き）
def create_vector_store(chunks):
    """ドキュメントチャンクからベクトルストアを作成（フォールバック機能付き）"""
    # 利用可能なモデルのリスト（優先順）
    model_options = [
        "models/text-embedding-004",          # 安定版（推奨）
        "models/gemini-embedding-exp-03-07",  # 最新の実験的モデル
        "models/text-embedding-005"           # 従来版
    ]

    for model_name in model_options:
        try:
            embeddings = GoogleGenerativeAIEmbeddings(model=model_name)
            vector_store = Chroma.from_documents(chunks, embeddings)
            print(f"成功: {model_name} を使用してベクトルストアを作成しました")
            return vector_store
        except Exception as e:
            print(f"警告: {model_name} でエラーが発生しました: {e}")
            continue

    raise Exception("すべてのEmbeddingモデルで失敗しました。API キーが正しく設定されているか確認してください。")

# LLMの初期化
def initialize_llm():
    """LLMを初期化"""
    try:
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        return llm
    except Exception as e:
        print(f"LLM初期化エラー: {e}")
        print("代替モデルを試行します...")
        try:
            return ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        except Exception as e2:
            raise Exception(f"LLMの初期化に失敗しました: {e2}")

# 方法1: 従来のStructuredOutputParserを使用（修正版）
def create_traditional_parser(llm):
    """従来のStructuredOutputParserを使用した分析チェーン"""
    # 重要条項抽出のスキーマ定義
    contract_schemas = [
        ResponseSchema(name="contract_period", description="契約の開始日と終了日"),
        ResponseSchema(name="payment_terms", description="支払い条件、金額、支払いスケジュール"),
        ResponseSchema(name="termination_clause", description="契約解除/解約の条件"),
        ResponseSchema(name="confidentiality", description="機密保持条項の詳細"),
        ResponseSchema(name="limitation_of_liability", description="責任の制限に関する条項"),
        ResponseSchema(name="governing_law", description="準拠法と管轄裁判所")
    ]

    # 抽出パーサーの設定
    parser = StructuredOutputParser.from_response_schemas(contract_schemas)

    # 条項抽出プロンプト
    extraction_prompt = PromptTemplate(
        template="""
        あなたは契約書分析の専門家です。以下の契約書の一部から、重要な条項を抽出してください。

        契約書:
        {text}

        以下の形式で重要な条項を抽出してください:
        {format_instructions}

        見つからない項目については "記載なし" と記述してください。
        """,
        input_variables=["text"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    # リスク分析プロンプト
    risk_analysis_prompt = PromptTemplate(
        template="""
        あなたは契約書のリスク分析専門家です。以下の契約条項を分析し、潜在的なリスクや不明確な点を指摘してください。

        抽出された条項:
        {extracted_clauses}

        各条項について以下の観点から分析してください:
        1. 不明確または曖昧な表現
        2. 一方に有利な条件
        3. 業界標準と比較した際の特異点
        4. 法的リスク

        具体的で実用的なアドバイスを提供してください。
        """,
        input_variables=["extracted_clauses"]
    )

    # 条項抽出チェーン
    extraction_chain = LLMChain(
        llm=llm,
        prompt=extraction_prompt,
        output_key="extracted_clauses"
    )

    # リスク分析チェーン
    risk_analysis_chain = LLMChain(
        llm=llm,
        prompt=risk_analysis_prompt,
        output_key="risk_analysis"
    )

    # 契約分析の全体チェーン
    contract_analysis_chain = SequentialChain(
        chains=[extraction_chain, risk_analysis_chain],
        input_variables=["text"],
        output_variables=["extracted_clauses", "risk_analysis"],
        verbose=True
    )

    return contract_analysis_chain

# 方法2: 現代的なwith_structured_outputを使用（推奨）
def create_modern_analyzer(llm):
    """現代的なwith_structured_outputを使用した分析"""

    # 構造化された条項抽出用のLLM
    contract_extractor = llm.with_structured_output(ContractAnalysis)

    # 構造化されたリスク分析用のLLM
    risk_analyzer = llm.with_structured_output(RiskAnalysis)

    # 構造化された比較分析用のLLM
    comparison_analyzer = llm.with_structured_output(ComparisonResult)

    return contract_extractor, risk_analyzer, comparison_analyzer

# 従来のアプローチを使用した分析関数
def analyze_contract_traditional(file_path, use_vector_store=True):
    """従来のStructuredOutputParserを使用した契約書分析"""
    try:
        print(f"契約書を読み込み中: {file_path}")
        docs = load_document(file_path)

        if use_vector_store:
            chunks = split_documents(docs)
            vector_store = create_vector_store(chunks)
            print("ベクトルストアを作成しました")

        # 全テキストを結合
        full_text = " ".join([doc.page_content for doc in docs])
        print(f"テキスト長: {len(full_text)} 文字")

        # LLMを初期化
        llm = initialize_llm()

        # 従来のチェーンを使用
        chain = create_traditional_parser(llm)
        results = chain.invoke({"text": full_text})

        return {
            "file_name": os.path.basename(file_path),
            "extracted_clauses": results["extracted_clauses"],
            "risk_analysis": results["risk_analysis"],
            "method": "traditional"
        }
    except Exception as e:
        print(f"分析中にエラーが発生しました: {e}")
        return {
            "file_name": os.path.basename(file_path),
            "error": str(e),
            "method": "traditional"
        }

# 現代的なアプローチを使用した分析関数（推奨）
def analyze_contract_modern(file_path, use_vector_store=True):
    """現代的なwith_structured_outputを使用した契約書分析"""
    try:
        print(f"契約書を読み込み中: {file_path}")
        docs = load_document(file_path)

        if use_vector_store:
            chunks = split_documents(docs)
            vector_store = create_vector_store(chunks)
            print("ベクトルストアを作成しました")

        # 全テキストを結合
        full_text = " ".join([doc.page_content for doc in docs])
        print(f"テキスト長: {len(full_text)} 文字")

        # LLMを初期化
        llm = initialize_llm()

        # 現代的なアプローチを使用
        contract_extractor, risk_analyzer, _ = create_modern_analyzer(llm)

        # 条項抽出
        extraction_prompt = f"""
        あなたは契約書分析の専門家です。以下の契約書から重要な条項を抽出してください。

        契約書:
        {full_text}

        契約期間、支払い条件、解約条項、機密保持、責任制限、準拠法について詳細に抽出してください。
        見つからない項目については None を返してください。
        """

        print("条項を抽出中...")
        extracted_clauses = contract_extractor.invoke(extraction_prompt)

        # リスク分析
        risk_prompt = f"""
        以下の契約条項を分析し、潜在的なリスクを特定してください。

        抽出された条項:
        {extracted_clauses}

        不明確な表現、一方的に有利な条件、業界標準からの逸脱、法的リスクを分析してください。
        具体的で実用的なアドバイスを提供してください。
        """

        print("リスク分析中...")
        risk_analysis = risk_analyzer.invoke(risk_prompt)

        return {
            "file_name": os.path.basename(file_path),
            "extracted_clauses": extracted_clauses,
            "risk_analysis": risk_analysis,
            "method": "modern"
        }
    except Exception as e:
        print(f"分析中にエラーが発生しました: {e}")
        return {
            "file_name": os.path.basename(file_path),
            "error": str(e),
            "method": "modern"
        }

# 複数の契約書を比較する関数
def compare_contracts(file_path1, file_path2, use_modern=True):
    """2つの契約書を比較し、相違点を報告"""
    try:
        print("契約書を比較中...")

        if use_modern:
            contract1 = analyze_contract_modern(file_path1)
            contract2 = analyze_contract_modern(file_path2)

            # LLMを初期化
            llm = initialize_llm()
            _, _, comparison_analyzer = create_modern_analyzer(llm)

            comparison_prompt = f"""
            以下の2つの契約書から抽出された条項を比較し、主要な相違点を報告してください。

            契約書1の条項:
            {contract1["extracted_clauses"]}

            契約書2の条項:
            {contract2["extracted_clauses"]}

            各条項の内容の違い、どちらの契約書がより有利な条件か、重要な条項の有無について分析してください。
            """

            comparison = comparison_analyzer.invoke(comparison_prompt)
        else:
            contract1 = analyze_contract_traditional(file_path1)
            contract2 = analyze_contract_traditional(file_path2)

            # 従来のプロンプトテンプレートを使用
            comparison_prompt = PromptTemplate(
                template="""
                あなたは契約書比較の専門家です。以下の2つの契約書から抽出された条項を比較し、主要な相違点を報告してください。

                契約書1の条項:
                {contract1_clauses}

                契約書2の条項:
                {contract2_clauses}

                以下の観点から比較分析を行ってください:
                1. 各条項の内容の違い
                2. どちらの契約書がより有利な条件か
                3. 重要な条項の有無
                """,
                input_variables=["contract1_clauses", "contract2_clauses"]
            )

            llm = initialize_llm()
            comparison_chain = LLMChain(llm=llm, prompt=comparison_prompt)

            comparison_result = comparison_chain.invoke({
                "contract1_clauses": contract1["extracted_clauses"],
                "contract2_clauses": contract2["extracted_clauses"]
            })
            comparison = comparison_result["text"]

        return {
            "contract1": contract1["file_name"],
            "contract2": contract2["file_name"],
            "comparison": comparison,
            "contract1_analysis": contract1,
            "contract2_analysis": contract2
        }
    except Exception as e:
        print(f"比較中にエラーが発生しました: {e}")
        return {
            "error": str(e)
        }

# バッチ処理機能
def analyze_multiple_contracts(file_paths, use_modern=True):
    """複数の契約書を一括分析"""
    results = []

    for file_path in file_paths:
        print(f"\n=== {file_path} の分析開始 ===")
        if use_modern:
            result = analyze_contract_modern(file_path)
        else:
            result = analyze_contract_traditional(file_path)
        results.append(result)
        print(f"=== {file_path} の分析完了 ===\n")

    return results

# 結果を整形して表示する関数
def print_analysis_results(result):
    """分析結果を見やすく表示"""
    print(f"\n{'='*50}")
    print(f"ファイル名: {result['file_name']}")
    print(f"分析方法: {result.get('method', '不明')}")
    print(f"{'='*50}")

    if 'error' in result:
        print(f"エラー: {result['error']}")
        return

    print("\n【抽出された条項】")
    print("-" * 30)
    if isinstance(result['extracted_clauses'], dict):
        for key, value in result['extracted_clauses'].items():
            print(f"{key}: {value}")
    else:
        print(result['extracted_clauses'])

    print("\n【リスク分析】")
    print("-" * 30)
    if isinstance(result['risk_analysis'], dict):
        for key, value in result['risk_analysis'].items():
            print(f"{key}: {value}")
    else:
        print(result['risk_analysis'])

# メイン実行関数
def main():
    """メイン実行関数"""
    print("契約書分析システムを開始します...")

    # 使用例1: 単一契約書の分析（現代的アプローチ）
    try:
        print("\n=== 単一契約書分析（現代的アプローチ） ===")
        contract_result = analyze_contract_modern("contract1.pdf")
        print_analysis_results(contract_result)

    except Exception as e:
        print(f"単一契約書分析でエラー: {e}")

    # 使用例2: 契約書比較
    try:
        print("\n=== 契約書比較 ===")
        comparison_result = compare_contracts("contract1.pdf", "contract2.pdf", use_modern=True)

        if 'error' not in comparison_result:
            print(f"比較対象: {comparison_result['contract1']} vs {comparison_result['contract2']}")
            print("\n【比較結果】")
            print("-" * 30)
            if isinstance(comparison_result['comparison'], dict):
                for key, value in comparison_result['comparison'].items():
                    print(f"{key}: {value}")
            else:
                print(comparison_result['comparison'])
        else:
            print(f"比較エラー: {comparison_result['error']}")

    except Exception as e:
        print(f"契約書比較でエラー: {e}")

    # 使用例3: 複数契約書の一括分析
    try:
        print("\n=== 複数契約書一括分析 ===")
        file_list = ["contract1.pdf", "contract2.pdf"]
        batch_results = analyze_multiple_contracts(file_list, use_modern=True)

        for result in batch_results:
            print_analysis_results(result)

    except Exception as e:
        print(f"一括分析でエラー: {e}")

# スクリプトとして実行された場合
if __name__ == "__main__":
    main()

契約書分析システムを開始します...

=== 単一契約書分析（現代的アプローチ） ===
契約書を読み込み中: contract1.pdf
分析中にエラーが発生しました: ファイルが見つかりません: contract1.pdf

ファイル名: contract1.pdf
分析方法: modern
エラー: ファイルが見つかりません: contract1.pdf

=== 契約書比較 ===
契約書を比較中...
契約書を読み込み中: contract1.pdf
分析中にエラーが発生しました: ファイルが見つかりません: contract1.pdf
契約書を読み込み中: contract2.pdf
分析中にエラーが発生しました: ファイルが見つかりません: contract2.pdf
比較中にエラーが発生しました: 'extracted_clauses'
比較エラー: 'extracted_clauses'

=== 複数契約書一括分析 ===

=== contract1.pdf の分析開始 ===
契約書を読み込み中: contract1.pdf
分析中にエラーが発生しました: ファイルが見つかりません: contract1.pdf
=== contract1.pdf の分析完了 ===


=== contract2.pdf の分析開始 ===
契約書を読み込み中: contract2.pdf
分析中にエラーが発生しました: ファイルが見つかりません: contract2.pdf
=== contract2.pdf の分析完了 ===


ファイル名: contract1.pdf
分析方法: modern
エラー: ファイルが見つかりません: contract1.pdf

ファイル名: contract2.pdf
分析方法: modern
エラー: ファイルが見つかりません: contract2.pdf


## 演習3：研究アシスタントエージェント
　あなたは、研究者向けのAIアシスタントを開発する企業のエンジニアです。以下の機能を持つ研究アシスタントエージェントを構築してください。

- 複数のツール（検索エンジン、論文データベース、計算ツールなど）を統合
- 研究テーマに関連する最新の論文や情報を検索・要約
- 研究データの分析支援
- 研究計画の立案と進捗管理
- リサーチノートの記録と整理

In [ ]:
!pip install -U duckduckgo-search==8.1.1
!pip install -U ddgs==9.6.1

In [ ]:
pip install arxiv==2.2.0

In [ ]:
!pip install langchain_experimental==0.3.4

In [ ]:
import os
import json # save_research_note と create_research_plan で使うためインポート
from datetime import datetime

# LangChain関連のインポート
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities.arxiv import ArxivAPIWrapper
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_core.tools import Tool, StructuredTool
from langchain.agents import AgentExecutor, create_tool_calling_agent # <- create_tool_calling_agent を使用
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 修正: pydantic を直接インポート
from pydantic import BaseModel, Field # Toolの引数スキーマ定義用
from langchain.memory import ConversationBufferMemory

# データ分析・可視化ライブラリ (Python REPL Tool内で使う可能性があるためインポートしておく)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Python実行ツールのインポート（エラーハンドリング付き）
try:
    from langchain_experimental.tools import PythonAstREPLTool # <- langchain_experimental からインポート
    PYTHON_REPL_AVAILABLE = True
    print("✓ PythonAstREPLTool が利用可能です")
except ImportError as e:
    print(f"⚠️ langchain_experimental が見つかりません: {e}")
    print("以下のコマンドでインストールしてください:")
    print("pip install langchain_experimental")
    PYTHON_REPL_AVAILABLE = False

# --- Google APIキーの設定 ---
# Colabを使用している場合：
# 左側の鍵アイコン（Secrets）で 'GOOGLE_API_KEY' という名前のシークレットを作成し、
# 自分のAPIキーを値として設定してください。
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

# ローカル環境の場合：
# 環境変数に GOOGLE_API_KEY を設定するか、以下のように直接設定します（非推奨）
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

# --- LLMの初期化 ---
def initialize_llm():
    """LLMを初期化（フォールバック機能付き）"""
    try:
        # gemini-2.5-flash は tool calling の性能が良い傾向があります
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            # temperature=0 # 必要に応じて温度を調整
            convert_system_message_to_human=True # GeminiはSystem Roleを直接サポートしないためHumanに変換
        )
        return llm
    except Exception as e:
        print(f"Gemini 1.5 Flash の初期化に失敗しました: {e}")
        print("代替モデルを試行します...")
        try:
            return ChatGoogleGenerativeAI(
                model="gemini-2.5-flash",
                convert_system_message_to_human=True
            )
        except Exception as e2:
            raise Exception(f"LLMの初期化に失敗しました: {e2}")

llm = initialize_llm()

# --- ツールの設定 ---

# 1. Web検索ツール
search_tool = DuckDuckGoSearchRun()

# 2. arXiv検索ツール
arxiv_api = ArxivAPIWrapper(
    top_k_results=3,              # 取得する論文数を減らす (API負荷軽減、応答速度改善)
    ARXIV_MAX_QUERY_LENGTH=300,
    load_max_docs=3,              # 読み込む論文数を減らす
    load_all_available_meta=False # 読み込むメタデータを最小限にする
)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_api)

# 3. Pythonコード実行ツール（条件付き）
if PYTHON_REPL_AVAILABLE:
    # pandasとnumpyをデフォルトで利用可能にする設定
    python_repl = PythonAstREPLTool(locals={"pd": pd, "np": np})
else:
    # 代替のPython実行機能（簡易版）
    def simple_python_exec(code: str) -> str:
        """簡易Python実行機能（langchain_experimental が利用できない場合の代替）"""
        try:
            # セキュリティ上の理由で、基本的な計算のみ許可
            allowed_modules = {'math', 'numpy', 'pandas'}

            # 安全なnamespaceを作成
            safe_globals = {
                "__builtins__": {
                    'abs': abs, 'max': max, 'min': min, 'sum': sum,
                    'len': len, 'range': range, 'enumerate': enumerate,
                    'zip': zip, 'map': map, 'filter': filter,
                    'print': print, 'str': str, 'int': int, 'float': float,
                    'list': list, 'dict': dict, 'tuple': tuple, 'set': set
                },
                'np': np,
                'pd': pd,
                'math': __import__('math')
            }

            # コードを実行
            exec_globals = safe_globals.copy()
            exec(code, exec_globals)

            return "コードが正常に実行されました。（簡易版Python実行機能を使用）"
        except Exception as e:
            return f"実行エラー: {str(e)}"

    # 簡易版ツールを作成
    class SimplePythonTool:
        def run(self, code: str) -> str:
            return simple_python_exec(code)

    python_repl = SimplePythonTool()

# 4. 研究ノート管理ツール
class ResearchNoteInput(BaseModel):
    """Input schema for save_research_note tool."""
    title: str = Field(..., description="研究ノートのタイトル")
    content: str = Field(..., description="研究ノートの内容")
    tags: str = Field(..., description="関連するタグ（カンマ区切り）")

def save_research_note(title: str, content: str, tags: str) -> str:
    """研究ノートを作成し、マークダウンファイルとして保存します。"""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"research_note_{title.replace(' ', '_')}_{timestamp}.md" # ファイル名にタイトルを含める

        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"# {title}\n\n")
            f.write(f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write(f"**Tags:** {tags}\n\n")
            f.write("---\n\n") # 区切り線
            f.write(f"{content}\n")

        return f"研究ノートを '{filename}' に正常に保存しました。"
    except Exception as e:
        return f"エラー：研究ノートの保存中に問題が発生しました: {str(e)}"

# 5. 研究計画管理ツール
class ResearchPlanInput(BaseModel):
    """Input schema for create_research_plan tool."""
    title: str = Field(..., description="研究計画のタイトル")
    objectives: str = Field(..., description="研究の目的と目標")
    methodology: str = Field(..., description="使用する方法論や手法")
    timeline: str = Field(..., description="研究のタイムライン、マイルストーン（例：1週目: 文献調査, 2-3週目: 実験準備...）")
    resources: str = Field(..., description="必要なリソースとツール（例：計算機資源、特定のソフトウェア、実験装置など）")

def create_research_plan(title: str, objectives: str, methodology: str, timeline: str, resources: str) -> str:
    """研究計画を作成し、マークダウンファイルとして保存します。"""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"research_plan_{title.replace(' ', '_')}_{timestamp}.md" # ファイル名にタイトルを含める

        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"# {title}\n\n")
            f.write(f"**作成日:** {datetime.now().strftime('%Y-%m-%d')}\n\n")
            f.write(f"## 1. 研究目的・目標\n\n{objectives}\n\n")
            f.write(f"## 2. 方法論\n\n{methodology}\n\n")
            f.write(f"## 3. タイムライン\n\n{timeline}\n\n")
            f.write(f"## 4. 必要リソース\n\n{resources}\n\n")

        return f"研究計画を '{filename}' に正常に保存しました。"
    except Exception as e:
        return f"エラー：研究計画の保存中に問題が発生しました: {str(e)}"

# 6. データ分析ツール（PythonREPLが使用できない場合の代替）
def analyze_data(data_description: str) -> str:
    """データ分析を行う代替機能"""
    try:
        # 簡単なデータ分析の例
        if "平均" in data_description or "average" in data_description.lower():
            return "データの平均値を計算するには、numpy.mean()またはpandas.Series.mean()を使用してください。"
        elif "可視化" in data_description or "plot" in data_description.lower():
            return "データの可視化には、matplotlib.pyplot.plot()やseaborn.lineplot()を使用してください。"
        else:
            return "より具体的なデータ分析要求を教えてください。例: '平均値の計算'、'データの可視化'など"
    except Exception as e:
        return f"データ分析エラー: {str(e)}"

# --- ツールリストの定義 ---
tools = [
    Tool(
        name="web_search",
        func=search_tool.run,
        description="インターネット上で一般的な情報や最新ニュースを検索します。検索クエリ（文字列）を入力してください。"
    ),
    Tool(
        name="arxiv_search",
        func=arxiv_tool.run,
        description="学術論文データベースarXivから論文を検索します。検索クエリ（論文タイトル、著者名、キーワードなど）を入力してください。"
    ),
    Tool(
        name="python_repl",
        func=python_repl.run,
        description=f"Pythonコードを実行してデータ分析、計算、簡単なプロット作成を行います。pandas (pd) と numpy (np) が利用可能です。実行したいPythonコード（文字列）を入力してください。{'（PythonAstREPLTool使用）' if PYTHON_REPL_AVAILABLE else '（簡易版実行機能使用）'}"
    ),
    # 修正: StructuredToolを使用
    StructuredTool(
        name="save_research_note",
        func=save_research_note,
        description="研究活動中に得られた知見やアイデアを記録するための研究ノートを作成・保存します。",
        args_schema=ResearchNoteInput
    ),
    # 修正: StructuredToolを使用
    StructuredTool(
        name="create_research_plan",
        func=create_research_plan,
        description="新しい研究プロジェクトの計画を作成・保存します。",
        args_schema=ResearchPlanInput
    )
]

# PythonREPLが利用できない場合の代替ツールを追加
if not PYTHON_REPL_AVAILABLE:
    tools.append(
        Tool(
            name="analyze_data",
            func=analyze_data,
            description="データ分析に関するアドバイスや簡単な分析を提供します。データの種類や分析したい内容を説明してください。"
        )
    )

# --- エージェントのプロンプト ---
# GeminiはSystem Roleを直接サポートしないため、Systemメッセージは内部的にHumanメッセージに変換される想定
# (ChatGoogleGenerativeAIの convert_system_message_to_human=True で対応)
# プロンプト内に {tools} は含めない
prompt = ChatPromptTemplate.from_messages([
    ("system", f"""あなたは、優秀で親切なAI研究アシスタントです。研究者の多様な学術活動（情報収集、文献調査、データ分析、アイデア整理、研究計画立案など）をサポートします。

指示されたタスクを実行するために、以下の利用可能なツールを適切に選択し、活用してください。
単に情報を提示するだけでなく、研究者の目的達成に貢献できるよう、分析的な視点や洞察を加えて応答することを心がけてください。

利用可能なツール:
- Web検索: 最新の情報や一般的な知識が必要な場合に使用します。
- arXiv検索: 特定の研究分野に関する学術論文を探す場合に使用します。
- Pythonコード実行: データ処理、計算、簡単な可視化が必要な場合に使用します。{'（PythonAstREPLTool使用）' if PYTHON_REPL_AVAILABLE else '（簡易版実行機能使用）'}
- 研究ノート保存: 重要な情報やアイデアを記録・整理したい場合に使用します。
- 研究計画作成: 新しい研究の骨子をまとめる場合に使用します。
{'- データ分析アドバイス: データ分析に関する一般的なアドバイスを提供します。' if not PYTHON_REPL_AVAILABLE else ''}

常に明確で、簡潔な言葉で応答してください。結果だけでなく、思考プロセスや使用したツールも示すことで、研究者が内容を理解しやすくなります。
以前の会話履歴（chat_history）を踏まえて、文脈に沿った応答をしてください。"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # エージェントの思考プロセスやツール使用履歴が入る
])

# --- メモリの設定 ---
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    # output_key="output" # AgentExecutorのデフォルト出力キーと合わせる (通常は不要)
    )

# --- エージェントの作成 ---
# GeminiのTool Callingに対応した `create_tool_calling_agent` を使用
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# --- エージェントエグゼキューターの作成 ---
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,               # 実行過程を詳細に表示
    handle_parsing_errors=True, # 出力解析エラー時にLLMに修正を促す試み
    max_iterations=10,          # 無限ループを防ぐための最大反復回数
    # early_stopping_method="generate", # LLMが最終回答を生成したと判断したら停止する
)

# --- 実行関数 ---
def research_assistant(query):
    """研究アシスタントエージェントを実行し、最終的な応答を返す"""
    try:
        response = agent_executor.invoke({"input": query})
        return response["output"]
    except Exception as e:
        print(f"エージェント実行中にエラーが発生しました: {e}")
        # handle_parsing_errors=True でも回復できない場合のエラーハンドリング
        # 必要に応じて、より詳細なエラー情報をログに出力するなどの処理を追加
        return "申し訳ありません、処理中にエラーが発生しました。"

# --- システム状態の確認 ---
def check_system_status():
    """システムの状態を確認"""
    print("=== 研究アシスタントシステム状態 ===")
    print(f"LLM: {llm.__class__.__name__}")
    print(f"Python REPL: {'✓ 利用可能' if PYTHON_REPL_AVAILABLE else '⚠️ 簡易版を使用'}")
    print(f"利用可能ツール数: {len(tools)}")
    print("ツール一覧:")
    for tool in tools:
        print(f"  - {tool.name}: {tool.description[:50]}...")
    print("=" * 40)

# --- 便利な関数 ---
def quick_search(query: str):
    """クイック検索（Webとarxivの両方を検索）"""
    print(f"🔍 '{query}' を検索中...")
    web_result = research_assistant(f"以下について検索してください: {query}")
    return web_result

def create_note(title: str, content: str, tags: str = "general"):
    """クイックノート作成"""
    return research_assistant(f"タイトル「{title}」、内容「{content}」、タグ「{tags}」で研究ノートを作成してください。")

def analyze_simple_data(data_description: str):
    """簡単なデータ分析"""
    if PYTHON_REPL_AVAILABLE:
        return research_assistant(f"以下のデータを分析してください: {data_description}")
    else:
        return research_assistant(f"以下のデータ分析についてアドバイスをください: {data_description}")

# --- テスト実行 ---
if __name__ == "__main__":
    # システム状態を確認
    check_system_status()

    print("\n🚀 研究アシスタントを開始します...\n")

    # 例1: Web検索とarXiv検索
    query1 = "最近のLLMにおける敵対的攻撃(adversarial attacks)に関する研究を調べて、主要な手法をいくつかリストアップしてください。"
    print(f"質問: {query1}")
    print(research_assistant(query1))
    print("\n" + "="*50 + "\n")

    # 例2: 研究ノートの保存
    query2 = "今日のミーティングで議論した「TransformerのAttention機構の計算コスト削減」に関するアイデアをノートにまとめたい。タイトルは「Attention計算効率化」、内容は「FlashAttentionやLinformerなどの近似手法の調査が必要」、タグは「LLM, Transformer, Attention」で保存して。"
    print(f"質問: {query2}")
    print(research_assistant(query2))
    print("\n" + "="*50 + "\n")

    # 例3: 研究計画の作成
    query3 = "「グラフニューラルネットワークを用いた材料特性予測」というテーマで研究計画を作成して。目的は高精度な予測モデル構築、方法は既存データセットでの学習と新規材料への適用、タイムラインは3ヶ月、リソースは計算サーバーとPythonライブラリ。"
    print(f"質問: {query3}")
    print(research_assistant(query3))
    print("\n" + "="*50 + "\n")

    # 例4: データ分析 (簡易)
    query4 = "簡単なデータ分析を試したい。[[1, 2], [3, 4], [5, 6]] というリストをnumpy配列に変換し、その平均値を計算して。"
    print(f"質問: {query4}")
    print(research_assistant(query4))
    print("\n" + "="*50 + "\n")

    # 最初の質問
    query_original = "量子コンピュータの最新研究動向について調べてください"
    print(f"質問: {query_original}")
    print(research_assistant(query_original))

    print("\n" + "="*50)
    print("💡 便利な関数の使用例:")
    print("- quick_search('機械学習')")
    print("- create_note('実験結果', '今日の実験で...', 'ML,実験')")
    print("- analyze_simple_data('売上データの月次推移')")

✓ PythonAstREPLTool が利用可能です
=== 研究アシスタントシステム状態 ===
LLM: ChatGoogleGenerativeAI
Python REPL: ✓ 利用可能
利用可能ツール数: 5
ツール一覧:
  - web_search: インターネット上で一般的な情報や最新ニュースを検索します。検索クエリ（文字列）を入力してください。...
  - arxiv_search: 学術論文データベースarXivから論文を検索します。検索クエリ（論文タイトル、著者名、キーワードなど...
  - python_repl: Pythonコードを実行してデータ分析、計算、簡単なプロット作成を行います。pandas (pd) ...
  - save_research_note: 研究活動中に得られた知見やアイデアを記録するための研究ノートを作成・保存します。...
  - create_research_plan: 新しい研究プロジェクトの計画を作成・保存します。...

🚀 研究アシスタントを開始します...

質問: 最近のLLMにおける敵対的攻撃(adversarial attacks)に関する研究を調べて、主要な手法をいくつかリストアップしてください。


> Entering new AgentExecutor chain...

Invoking: `arxiv_search` with `LLM adversarial attacks`
responded: はい、承知いたしました。LLMにおける敵対的攻撃に関する主要な研究手法について調査し、リストアップします。

まず、arXivで関連する学術論文を検索します。



Published: 2024-07-04
Title: Is LLM-as-a-Judge Robust? Investigating Universal Adversarial Attacks on Zero-shot LLM Assessment
Authors: Vyas Raina, Adian Liusie, Mark Gales
Summary: Large Language Models (LLMs) are powerful zero-shot assessors used in
real-wor

## 演習4：マルチステップ意思決定エージェント
　製品開発プロセスをサポートするAIエージェントを開発してください。この
エージェントは、製品アイデアからプロトタイプまでの複数ステップを管理し、
各ステップで適切な意思決定を行う必要があります。具体的には以下の機能を実
装してください。

- アイデア生成と評価
- 市場調査と競合分析
- 製品仕様の定義
- プロトタイプの計画
- フィードバックの収集と分析

In [ ]:
# ライブラリインポート
import os
from typing import List, Dict, Any, TypedDict, Optional # Optionalを追加
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage # BaseMessageを追加
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai.types as safety_types
# Gemini API固有のエラーをインポート
from google.generativeai.types.generation_types import BlockedPromptException, StopCandidateException
from google.api_core import exceptions as google_api_exceptions # google-api-coreの例外を追加
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type, RetryCallState
import json
import operator
import re
import pprint
import logging # loggingを追加

# ロギング設定 (より詳細な情報を出力)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- 環境変数設定 (APIキー) ---
# GOOGLE_API_KEY = "YOUR_API_KEY" # 必要に応じて設定
# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# --- LLMの初期化 ---
logger.warning("!!! WARNING: Safety settings are temporarily disabled for debugging. !!!")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # または gemini-2.5-pro
    safety_settings={
        safety_types.HarmCategory.HARM_CATEGORY_HARASSMENT: safety_types.HarmBlockThreshold.BLOCK_NONE,
        safety_types.HarmCategory.HARM_CATEGORY_HATE_SPEECH: safety_types.HarmBlockThreshold.BLOCK_NONE,
        safety_types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: safety_types.HarmBlockThreshold.BLOCK_NONE,
        safety_types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: safety_types.HarmBlockThreshold.BLOCK_NONE,
    },
    # JSONモード指定は維持
    generation_config={"response_mime_type": "application/json"},
    # temperature=0.7
)

# --- リトライ付きLLM呼び出し関数 (エラーハンドリング強化) ---
class EmptyResponseError(Exception): pass

# リトライ対象の例外を定義 (APIエラー、サービス不可、タイムアウト、空応答など)
RETRYABLE_EXCEPTIONS = (
    google_api_exceptions.ResourceExhausted, # Rate limitなど
    google_api_exceptions.ServiceUnavailable, # サーバー側の一時的な問題
    google_api_exceptions.DeadlineExceeded, # タイムアウト
    google_api_exceptions.Aborted, # リクエスト中断
    google_api_exceptions.InternalServerError, # サーバー内部エラー
    EmptyResponseError, # カスタム例外: 空応答
    BlockedPromptException, # SAFETY以外の理由でブロックされた場合 (リトライする価値があるか要検討)
    # その他の予期せぬエラーも念のためリトライ対象に入れる場合は Exception を追加
    # Exception
)

# リトライ試行時のログ出力関数
def log_retry_attempt(retry_state: RetryCallState):
    attempt_num = retry_state.attempt_number
    exception = retry_state.outcome.exception()
    logger.warning(f"LLM call failed (attempt {attempt_num}). Retrying in {retry_state.next_action.sleep} seconds. Error: {type(exception).__name__}: {exception}")

@retry(
    stop=stop_after_attempt(3),
    wait=wait_fixed(3),
    retry=retry_if_exception_type(RETRYABLE_EXCEPTIONS),
    after=log_retry_attempt # リトライ試行後にログ出力
)
def invoke_llm_with_retry(llm_instance: ChatGoogleGenerativeAI, messages_input: List[BaseMessage]) -> AIMessage: # 型ヒントをBaseMessageに修正
    attempt_num = getattr(invoke_llm_with_retry.retry.statistics, 'attempt_number', 1) # tenacity v8以降
    logger.info(f"Attempting LLM call (attempt {attempt_num})...")
    try:
        # --- 入力メッセージのログ出力 (デバッグ用、長すぎる場合は注意) ---
        # logger.debug("LLM Input Messages:")
        # for i, msg in enumerate(messages_input):
        #     content_preview = str(getattr(msg, 'content', ''))[:200] + ('...' if len(str(getattr(msg, 'content', ''))) > 200 else '')
        #     logger.debug(f"  [{i}] {type(msg).__name__}: {content_preview}")
        # ----------------------------------------------------------------

        total_chars = sum(len(str(getattr(m, 'content', ''))) for m in messages_input)
        logger.info(f"Input messages total length (approx chars): {total_chars}")
        if total_chars > 500000: logger.warning("Warning: Input message length is very large.")
    except Exception as e: logger.warning(f"Warning: Could not calculate message length - {e}")

    try:
        response = llm_instance.invoke(messages_input)
    except BlockedPromptException as bpe:
        # プロンプト自体がブロックされた場合 (Safety以外も含む)
        logger.error(f"LLM call blocked by API (attempt {attempt_num}): {bpe}")
        raise # リトライ対象ならリトライされる
    except StopCandidateException as sce:
         # Safety Settings などで候補が停止した場合 (BLOCK_NONEでも発生しうる?)
         finish_reason = getattr(sce, 'finish_reason', 'UNKNOWN') # finish_reasonを取得試行
         logger.error(f"LLM candidate stopped (attempt {attempt_num}). Finish Reason: {finish_reason}. Error: {sce}")
         # BLOCK_NONE設定でSAFETYが理由の場合は特に警告
         if finish_reason == "SAFETY":
             logger.critical(f"LLM response blocked by SAFETY unexpectedly (attempt {attempt_num}) despite BLOCK_NONE settings!")
             # ここで特定の処理（例：フォールバックを強制）を行うか、例外をそのまま投げるか選択
             # 今回は例外を投げてリトライまたは上位のexceptで捕捉させる
         raise # リトライ対象ならリトライされる
    except google_api_exceptions.ApiException as api_exc:
        # その他のGoogle APIエラー
        logger.error(f"Google API error during LLM call (attempt {attempt_num}): {type(api_exc).__name__} - {api_exc}")
        raise # リトライ対象ならリトライされる
    except Exception as e:
        # その他の予期せぬエラー
        logger.error(f"Unexpected error during LLM invocation (attempt {attempt_num}): {type(e).__name__} - {e}", exc_info=True) # トレースバックも記録
        raise # リトライ対象として設定していればリトライされる

    # レスポンス内容のチェック
    if not response.content or not str(response.content).strip():
        logger.warning(f"LLM returned empty content unexpectedly on attempt {attempt_num}, raising error for retry...")
        raise EmptyResponseError("LLM returned empty or whitespace content.")

    # Safety check (念のため再度確認) - StopCandidateException で捕捉されるはずだが二重チェック
    finish_reason = response.response_metadata.get("finish_reason")
    if finish_reason == "SAFETY":
        logger.critical(f"LLM call stopped due to SAFETY (attempt {attempt_num}) despite BLOCK_NONE settings and previous checks. This indicates an unexpected API behavior.")
        # この時点でエラーとする (リトライは通常しない)
        raise Exception("LLM response blocked by safety filter unexpectedly after successful invocation.")

    logger.info(f"LLM call attempt {attempt_num} successful.")
    return response

# 製品開発プロセスの状態を定義 (変更なし)
class ProductDevelopmentState(TypedDict):
    messages: List[BaseMessage] # 型ヒントをBaseMessageに修正
    current_step: Optional[str] # Optionalを追加
    product_idea: Dict[str, Any]
    market_research: Dict[str, Any]
    product_spec: Dict[str, Any]
    prototype_plan: Dict[str, Any]
    feedback: List[Dict[str, Any]]

# --- ユーティリティ関数: LLM応答からJSONを抽出 (ログ追加) ---
def extract_json_from_response(response_content: str, node_name: str) -> Optional[Dict[str, Any]]:
    """
    LLM応答からJSONを抽出・パースする。
    JSONモードでも ```json ``` ブロックが付与される場合と、JSON文字列のみの場合に対応。
    """
    if not response_content:
        logger.warning(f"Warning: (extract_json) Received empty content in {node_name}.")
        return None

    logger.debug(f"LLM Response ({node_name}) to be parsed:\n{response_content[:1000]}...") # 全文表示は避ける
    json_str_to_parse = None
    try:
        # まず ```json ``` ブロックを探す
        json_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', response_content, re.DOTALL | re.IGNORECASE)
        if json_match:
            # ブロックがあればその中身をパース対象とする
            json_str_to_parse = json_match.group(1).strip()
            logger.info(f"Found ```json ``` block in {node_name}. Extracted content for parsing.")
            logger.debug(f"Extracted JSON string (first 500 chars): {json_str_to_parse[:500]}")
        else:
            # ブロックがなければ応答全体をパース対象とする (JSONモードの本来の動作)
            logger.info(f"No ```json ``` block found in {node_name}. Assuming the whole content is JSON.")
            json_str_to_parse = response_content.strip()

        # パース対象文字列があればパース試行
        if json_str_to_parse:
            parsed_json = json.loads(json_str_to_parse)
            logger.info(f"Successfully parsed JSON from response in {node_name}.")
            return parsed_json
        else:
            # ```json```ブロックも無く、応答全体も空だった場合
            logger.warning(f"Warning: (extract_json) No content to parse after extraction/stripping in {node_name}.")
            return None

    except json.JSONDecodeError as e:
        target = "block content" if json_match else "whole content"
        logger.error(f"Error: Failed decoding {target} as JSON in {node_name}: {e}")
        logger.debug(f"Content that failed parsing in {node_name} (first 1000 chars): {json_str_to_parse[:1000] if json_str_to_parse else 'N/A'}")
        return None
    except Exception as e:
        logger.error(f"An unexpected error occurred during JSON extraction/parsing in {node_name}: {e}", exc_info=True)
        return None

# --- フォールバック用データ生成関数 (ログ追加) ---
def get_fallback_data(node_name: str, error_content: str = "Error during generation") -> Dict[str, Any]:
    """指定されたノードに対応するフォールバックデータを返す"""
    logger.warning(f"Generating fallback data for node '{node_name}' due to error: {error_content[:200]}")
    # エラー発生ノード名をエラー情報に追加
    fallback = {"error_info": f"Fallback data used in [{node_name}] due to: {error_content[:200]}..."}
    if node_name == "generate_idea":
        fallback.update({ "product_name": "未定義 (フォールバック)", "description": "アイデア生成失敗", "unique_value": "-", "target_audience": "-", "feasibility_score": 0, "innovation_score": 0, "market_potential": 0 })
    elif node_name == "market_research":
        fallback.update({ "market_size": "-", "target_segments": [], "competitors": [], "entry_barriers": [], "competitive_advantage": [], "threats": [] })
    elif node_name == "define_product_spec":
         fallback.update({ "core_features": [], "technical_requirements": "定義失敗", "performance_metrics": [], "user_experience": "定義失敗", "integration_points": [], "constraints": {}, "success_criteria": [] })
    elif node_name == "plan_prototype":
         fallback.update({ "prototype_type": "計画失敗", "scope": {"included": [], "excluded": []}, "testing_objectives": [], "resources_needed": {}, "timeline": [], "success_metrics": [], "risk_mitigation": [] })
    elif node_name == "collect_feedback":
         fallback.update({ "target_stakeholders": [], "feedback_methods": [], "key_questions": {}, "data_analysis_plan": "計画失敗", "potential_insights": [], "action_plan_framework": "計画失敗", "iteration_strategy": "計画失敗" })
    else: fallback["error"] = error_content
    return fallback

# --- ノード関数 (エラーハンドリングとログ改善) ---

def generate_idea(state: ProductDevelopmentState) -> ProductDevelopmentState:
    node_name = "generate_idea"
    logger.info(f"\n--- Running {node_name} ---")
    messages = state["messages"]
    # JSONモード用プロンプト (簡略化)
    system_prompt = SystemMessage(content="""あなたは製品アイデア生成のエキスパートです。ユーザーの初期要求に基づいて、革新的で実現可能な製品アイデアを1つ提案してください。以下のJSON形式で出力してください:
{
  "product_name": "製品名",
  "description": "製品の簡単な説明 (1-2文)",
  "unique_value": "独自の価値提案、競合との差別化要因",
  "target_audience": "主なターゲット顧客層",
  "feasibility_score": "実現可能性スコア (1-10、10が最高)",
  "innovation_score": "革新性スコア (1-10、10が最高)",
  "market_potential": "市場潜在力スコア (1-10、10が最高)"
}
必ず上記のJSON形式のみを出力してください。他のテキストは含めないでください。""")

    response_content = ""
    product_idea = None
    response_message = None
    fallback_error_content = "Unknown error"
    try:
        # 呼び出し入力: System + History
        input_messages = [system_prompt] + messages
        response = invoke_llm_with_retry(llm, input_messages)
        response_content = response.content
        response_message = response
        product_idea = extract_json_from_response(response_content, node_name)
        if product_idea is None:
             fallback_error_content = "JSON parsing failed from LLM response" if response_content else "LLM returned empty content"

    except Exception as e:
        logger.error(f"Error in {node_name} during LLM call or processing: {type(e).__name__} - {e}", exc_info=True)
        fallback_error_content = f"{type(e).__name__}: {e}"
        response_message = None # エラー時はレスポンスメッセージを使わない

    if product_idea is None:
        logger.warning(f"Using fallback data for {node_name} due to: {fallback_error_content}")
        product_idea = get_fallback_data(node_name, fallback_error_content)

    # --- 状態更新、案内メッセージ生成 ---
    new_state = state.copy()
    new_state["product_idea"] = product_idea
    current_messages = list(messages) # 変更可能なリストにする

    # 成功したLLM応答（JSONパース成否問わず）があれば履歴に追加
    if response_message and response_message.content and str(response_message.content).strip():
        logger.info(f"Adding LLM response from {node_name} to messages.")
        current_messages.append(response_message)
    elif response_message:
        logger.warning(f"Skipping adding empty/whitespace LLM response from {node_name} to messages.")

    # 案内メッセージ生成
    if "error_info" not in product_idea:
        next_step_message_content = f"""製品アイデア「{product_idea.get('product_name', '名称未定')}」を生成しました。
概要: {product_idea.get('description', '説明なし')}
ユニークな価値: {product_idea.get('unique_value', '未定義')}
次に、このアイデアに基づいた市場調査を行います。"""
    else:
        error_detail = product_idea.get("error_info", "不明なエラー")
        next_step_message_content = f"""製品アイデアの生成に失敗しました (理由: {error_detail[:100]}...)。フォールバックデータを使用します。
次に、仮のデータで市場調査ステップに進みます。"""
    next_step_message = AIMessage(content=next_step_message_content)
    current_messages.append(next_step_message)

    new_state["messages"] = current_messages
    new_state["current_step"] = node_name
    logger.info(f"Updated state in {node_name}. current_step set to '{node_name}'.")
    return new_state

# market_research, define_product_spec, plan_prototype, collect_feedback も同様の構造で修正
# (エラーハンドリング、ロギング、状態更新ロジックを上記 generate_idea に合わせる)

def market_research(state: ProductDevelopmentState) -> ProductDevelopmentState:
    node_name = "market_research"
    logger.info(f"\n--- Running {node_name} ---")
    messages = state["messages"] # 全履歴を使う
    product_idea = state["product_idea"]
    # JSONモード用プロンプト (簡略化)
    system_prompt = SystemMessage(content=f"""あなたは市場調査の専門家です。提供された製品アイデアに基づいて市場調査を行い、以下のJSON形式で結果をまとめてください:
製品アイデア: {json.dumps(product_idea, ensure_ascii=False, indent=2)}
出力形式:
{{
  "market_size": "推定市場規模 (例: '約10億円', '年間100万ユニット')",
  "target_segments": [ {{ "segment": "ターゲット顧客セグメント名", "needs": "主なニーズや課題" }} ],
  "competitors": [ {{ "name": "競合製品/企業名", "strengths": "強み", "weaknesses": "弱み" }} ],
  "entry_barriers": [ "参入障壁 (例: '高い開発コスト', 'ブランド認知度')" ],
  "competitive_advantage": [ "自社製品の競争優位性" ],
  "threats": [ "市場の脅威 (例: '技術の急速な変化', '価格競争')" ]
}}
必ず上記のJSON形式のみを出力してください。他のテキストは含めないでください。""")

    response_content = ""
    market_research_data = None
    response_message = None
    fallback_error_content = "Unknown error"
    try:
        # 呼び出し入力: System + History
        input_messages = [system_prompt] + messages
        response = invoke_llm_with_retry(llm, input_messages)
        response_content = response.content
        response_message = response
        market_research_data = extract_json_from_response(response_content, node_name)
        if market_research_data is None:
            fallback_error_content = "JSON parsing failed from LLM response" if response_content else "LLM returned empty content"

    except Exception as e:
        logger.error(f"Error in {node_name} during LLM call or processing: {type(e).__name__} - {e}", exc_info=True)
        fallback_error_content = f"{type(e).__name__}: {e}"
        response_message = None

    if market_research_data is None:
        logger.warning(f"Using fallback data for {node_name} due to: {fallback_error_content}")
        market_research_data = get_fallback_data(node_name, fallback_error_content)

    # --- 状態更新、案内メッセージ生成 ---
    new_state = state.copy()
    new_state["market_research"] = market_research_data
    current_messages = list(messages)

    if response_message and response_message.content and str(response_message.content).strip():
        logger.info(f"Adding LLM response from {node_name} to messages.")
        current_messages.append(response_message)
    elif response_message:
         logger.warning(f"Skipping adding empty/whitespace LLM response from {node_name} to messages.")

    if "error_info" not in market_research_data:
        next_step_message_content = f"""市場調査が完了しました。
市場規模: {market_research_data.get('market_size', '不明')}
ターゲットセグメント数: {len(market_research_data.get('target_segments', []))}
競合数: {len(market_research_data.get('competitors', []))}
次に、これらの情報をもとに製品仕様を定義します。"""
    else:
        error_detail = market_research_data.get("error_info", "不明なエラー")
        next_step_message_content = f"""市場調査に失敗しました (理由: {error_detail[:100]}...)。フォールバックデータを使用します。
次に、仮のデータで製品仕様定義ステップに進みます。"""
    next_step_message = AIMessage(content=next_step_message_content)
    current_messages.append(next_step_message)

    new_state["messages"] = current_messages
    new_state["current_step"] = node_name
    logger.info(f"Updated state in {node_name}. current_step set to '{node_name}'.")
    return new_state

def define_product_spec(state: ProductDevelopmentState) -> ProductDevelopmentState:
    node_name = "define_product_spec"
    logger.info(f"\n--- Running {node_name} ---")
    messages = state["messages"] # 全履歴を使う
    product_idea = state["product_idea"]
    market_research = state["market_research"]
    # JSONモード用プロンプト (簡略化)
    system_prompt = SystemMessage(content=f"""あなたはプロダクトマネージャーです。製品アイデアと市場調査結果に基づき、詳細な製品仕様を定義してください。以下のJSON形式で出力してください:
製品アイデア: {json.dumps(product_idea.get("product_name", "N/A"), ensure_ascii=False)}
市場調査サマリ: 市場規模={market_research.get("market_size", "N/A")}, ターゲット={len(market_research.get("target_segments",[]))}セグメント, 競合={len(market_research.get("competitors",[]))}社
出力形式:
{{
  "core_features": [ {{ "feature": "主要機能名", "description": "機能概要", "priority": "High/Medium/Low" }} ],
  "technical_requirements": "技術要件 (例: 'Webアプリ、Pythonバックエンド、Reactフロントエンド')",
  "performance_metrics": [ {{ "metric": "性能指標名", "target": "目標値" }} ],
  "user_experience": {{ "ui_ux_principles": "UI/UX設計原則", "key_user_flows": ["主要なユーザーフロー1", "..."] }},
  "integration_points": [ "連携が必要な外部システムやAPI" ],
  "constraints": {{ "budget": "予算制約", "timeline": "期間制約", "technology": "技術制約" }},
  "success_criteria": [ "製品の成功を測る基準" ]
}}
必ず上記のJSON形式のみを出力してください。他のテキストは含めないでください。""")

    response_content = ""
    product_spec = None
    response_message = None
    fallback_error_content = "Unknown error"
    try:
        # 呼び出し入力: System + History
        input_messages = [system_prompt] + messages
        response = invoke_llm_with_retry(llm, input_messages)
        response_content = response.content
        response_message = response
        product_spec = extract_json_from_response(response_content, node_name)
        if product_spec is None:
            fallback_error_content = "JSON parsing failed from LLM response" if response_content else "LLM returned empty content"

    except Exception as e:
        logger.error(f"Error in {node_name} during LLM call or processing: {type(e).__name__} - {e}", exc_info=True)
        fallback_error_content = f"{type(e).__name__}: {e}"
        response_message = None

    if product_spec is None:
        logger.warning(f"Using fallback data for {node_name} due to: {fallback_error_content}")
        product_spec = get_fallback_data(node_name, fallback_error_content)

    # --- 状態更新、案内メッセージ生成 ---
    new_state = state.copy()
    new_state["product_spec"] = product_spec
    current_messages = list(messages)

    if response_message and response_message.content and str(response_message.content).strip():
        logger.info(f"Adding LLM response from {node_name} to messages.")
        current_messages.append(response_message)
    elif response_message:
         logger.warning(f"Skipping adding empty/whitespace LLM response from {node_name} to messages.")

    if "error_info" not in product_spec:
        core_features_count = len(product_spec.get('core_features', []))
        next_step_message_content = f"""製品仕様の定義が完了しました。
主要機能数: {core_features_count}
技術要件: {product_spec.get('technical_requirements', '未定義')}
次に、この仕様に基づいてプロトタイプ開発計画を作成します。"""
    else:
        error_detail = product_spec.get("error_info", "不明なエラー")
        next_step_message_content = f"""製品仕様の定義に失敗しました (理由: {error_detail[:100]}...)。フォールバックデータを使用します。
次に、仮のデータでプロトタイプ計画ステップに進みます。"""
    next_step_message = AIMessage(content=next_step_message_content)
    current_messages.append(next_step_message)

    new_state["messages"] = current_messages
    new_state["current_step"] = node_name
    logger.info(f"Updated state in {node_name}. current_step set to '{node_name}'.")
    return new_state

def plan_prototype(state: ProductDevelopmentState) -> ProductDevelopmentState:
    node_name = "plan_prototype"
    logger.info(f"\n--- Running {node_name} ---")
    messages = state["messages"] # 全履歴を使う
    product_spec = state["product_spec"]
    product_idea = state["product_idea"] # 参考情報として
    # JSONモード用プロンプト (簡略化)
    system_prompt = SystemMessage(content=f"""あなたは開発リーダーです。提供された製品仕様に基づき、プロトタイプ開発計画を作成してください。特に優先度の高い主要機能に焦点を当ててください。以下のJSON形式で出力してください:
製品名: {product_idea.get("product_name", "N/A")}
主要機能({len(product_spec.get('core_features',[]))}件): {json.dumps([f['feature'] for f in product_spec.get('core_features',[])[:3]], ensure_ascii=False)}...
出力形式:
{{
  "prototype_type": "プロトタイプの種類 (例: 'クリック可能モックアップ', '機能限定PoC', 'ワイヤーフレーム')",
  "scope": {{ "included": ["プロトタイプに含める主要機能や要素"], "excluded": ["スコープ外の機能"] }},
  "testing_objectives": ["プロトタイプで検証したいこと"],
  "resources_needed": {{ "personnel": ["必要な役割 (例: 'デザイナー1名', 'エンジニア1名')"], "tools": ["使用ツール (例: 'Figma', 'React')"] }},
  "timeline": [ {{ "phase": "フェーズ名 (例: '設計', '開発', 'テスト')", "duration_weeks": "期間 (週)" }} ],
  "success_metrics": ["プロトタイプ成功の測定基準 (例: '主要なユーザータスクの完了率')"],
  "risk_mitigation": [ {{ "risk": "想定されるリスク", "mitigation": "対応策" }} ]
}}
必ず上記のJSON形式のみを出力してください。他のテキストは含めないでください。""")

    response_content = ""
    prototype_plan = None
    response_message = None
    fallback_error_content = "Unknown error"
    try:
        # 呼び出し入力: System + History
        input_messages = [system_prompt] + messages
        response = invoke_llm_with_retry(llm, input_messages)
        response_content = response.content
        response_message = response
        prototype_plan = extract_json_from_response(response_content, node_name)
        if prototype_plan is None:
            fallback_error_content = "JSON parsing failed from LLM response" if response_content else "LLM returned empty content"

    except Exception as e:
        logger.error(f"Error in {node_name} during LLM call or processing: {type(e).__name__} - {e}", exc_info=True)
        fallback_error_content = f"{type(e).__name__}: {e}"
        response_message = None

    if prototype_plan is None:
        logger.warning(f"Using fallback data for {node_name} due to: {fallback_error_content}")
        prototype_plan = get_fallback_data(node_name, fallback_error_content)

    # --- 状態更新、案内メッセージ生成 ---
    new_state = state.copy()
    new_state["prototype_plan"] = prototype_plan
    current_messages = list(messages)

    if response_message and response_message.content and str(response_message.content).strip():
        logger.info(f"Adding LLM response from {node_name} to messages.")
        current_messages.append(response_message)
    elif response_message:
        logger.warning(f"Skipping adding empty/whitespace LLM response from {node_name} to messages.")

    if "error_info" not in prototype_plan:
        prototype_type = prototype_plan.get('prototype_type', '未定')
        timeline_weeks = sum(phase.get('duration_weeks', 0) for phase in prototype_plan.get('timeline', []))
        next_step_message_content = f"""プロトタイプ開発計画の作成が完了しました。
プロトタイプ種類: {prototype_type}
想定期間: 約{timeline_weeks}週間
次に、この計画に基づいてフィードバック収集計画を立てます。"""
    else:
        error_detail = prototype_plan.get("error_info", "不明なエラー")
        next_step_message_content = f"""プロトタイプ開発計画の作成に失敗しました (理由: {error_detail[:100]}...)。フォールバックデータを使用します。
次に、仮のデータでフィードバック収集計画ステップに進みます。"""
    next_step_message = AIMessage(content=next_step_message_content)
    current_messages.append(next_step_message)

    new_state["messages"] = current_messages
    new_state["current_step"] = node_name
    logger.info(f"Updated state in {node_name}. current_step set to '{node_name}'.")
    return new_state

def collect_feedback(state: ProductDevelopmentState) -> ProductDevelopmentState:
    node_name = "collect_feedback"
    logger.info(f"\n--- Running {node_name} ---")
    messages = state["messages"] # 全履歴を使う
    product_idea = state["product_idea"]
    prototype_plan = state["prototype_plan"]
    market_research = state["market_research"]
    product_spec = state["product_spec"]

    # --- プロンプト用のサマリ生成 (エラーハンドリング改善) ---
    try:
        # 安全なgetとデフォルト値、リスト/辞書の存在確認を追加
        product_idea_summary = { k: product_idea.get(k, "N/A") for k in ["product_name", "description", "unique_value"] } if product_idea else {"error": "Product Idea missing"}

        mr_target_segments = market_research.get("target_segments", []) if market_research else []
        mr_competitors = market_research.get("competitors", []) if market_research else []
        market_research_summary = {
            "target_segments": [seg.get("segment", "N/A") for seg in mr_target_segments[:3]],
            "competitors": [comp.get("name", "N/A") for comp in mr_competitors[:3]]
        }

        ps_core_features = product_spec.get("core_features", []) if product_spec else []
        ps_user_exp = product_spec.get("user_experience", {}) if product_spec else {}
        product_spec_summary = {
             "core_features_high_priority": [feat.get("feature", "N/A") for feat in ps_core_features if isinstance(feat, dict) and feat.get("priority") == "High"][:5],
             "user_experience_summary": ps_user_exp.get("ui_ux_principles", "N/A") if isinstance(ps_user_exp, dict) else "N/A"
        }

        prototype_plan_summary_str = "N/A"
        if prototype_plan and "error_info" not in prototype_plan:
            pt_testing_objectives = prototype_plan.get("testing_objectives", [])
            prototype_plan_summary = {
                "prototype_type": prototype_plan.get("prototype_type", "N/A"),
                "testing_objectives": pt_testing_objectives[:3]
            }
            prototype_plan_summary_str = json.dumps(prototype_plan_summary, ensure_ascii=False, indent=2)
        elif prototype_plan and "error_info" in prototype_plan:
             logger.warning(f"Using fallback prototype_plan for {node_name} prompt.")
             prototype_plan_summary_str = "{ /* プロトタイプ計画にエラーあり */ }"
        else:
             logger.warning(f"Prototype plan is missing or invalid for {node_name} prompt.")
             prototype_plan_summary_str = "{ /* プロトタイプ計画なし */ }"

    except Exception as e:
        logger.error(f"Error creating summary for {node_name} prompt: {e}", exc_info=True)
        product_idea_summary, market_research_summary, product_spec_summary = {}, {}, {}
        prototype_plan_summary_str = "{ /* サマリ生成エラー */ }"

    # JSONモード用プロンプト (簡略化)
    system_prompt = SystemMessage(content=f"""あなたはリサーチプランナーです。これまでの製品開発情報（アイデア、市場調査、仕様、プロトタイプ計画）に基づき、効果的なフィードバック収集計画を立ててください。以下のJSON形式で出力してください:
製品アイデアサマリ: {json.dumps(product_idea_summary, ensure_ascii=False)}
市場調査サマリ: {json.dumps(market_research_summary, ensure_ascii=False)}
製品仕様サマリ: {json.dumps(product_spec_summary, ensure_ascii=False)}
プロトタイプ計画サマリ: {prototype_plan_summary_str}
出力形式:
{{
  "target_stakeholders": ["フィードバックを得たい対象者 (例: 'ターゲット顧客セグメントAのユーザー5名', '社内デザイナー')"],
  "feedback_methods": ["具体的な収集方法 (例: 'ユーザビリティテスト', 'インタビュー', 'アンケート')"],
  "key_questions": {{ "value_prop": ["価値提案に関する質問"], "usability": ["使いやすさに関する質問"], "features": ["特定の機能に関する質問"] }},
  "data_analysis_plan": "収集したデータの分析方法",
  "potential_insights": ["期待される主な洞察"],
  "action_plan_framework": "フィードバックを次に活かすための枠組み (例: '優先度付けマトリクス作成', '次期イテレーションへの反映')",
  "iteration_strategy": "フィードバック後の反復戦略 (例: '結果に基づきプロトタイプを修正し再テスト', '仕様へフィードバック')"
}}
必ず上記のJSON形式のみを出力してください。他のテキストは含めないでください。""")

    response_content = ""
    feedback_plan = None
    response_message = None
    fallback_error_content = "Unknown error"
    try:
        # LLM呼び出し: システムプロンプト + 有効な全履歴
        valid_messages = []
        for msg in messages:
            msg_content = getattr(msg, 'content', None)
            # SystemMessageはそのまま、他は内容が空でないことを確認
            if isinstance(msg, SystemMessage) or (msg_content is not None and str(msg_content).strip()):
                valid_messages.append(msg)
            else:
                logger.warning(f"Filtering out message with empty or invalid content before {node_name}: {type(msg)}")
        input_messages = [system_prompt] + valid_messages

        response = invoke_llm_with_retry(llm, input_messages)
        response_content = response.content
        response_message = response
        feedback_plan = extract_json_from_response(response_content, node_name)
        if feedback_plan is None:
            fallback_error_content = "JSON parsing failed from LLM response" if response_content else "LLM returned empty content"

    except Exception as e:
        logger.error(f"Error in {node_name} during LLM call or processing: {type(e).__name__} - {e}", exc_info=True)
        fallback_error_content = f"{type(e).__name__}: {e}"
        response_message = None

    if feedback_plan is None:
        logger.warning(f"Using fallback data for {node_name} due to: {fallback_error_content}")
        feedback_plan = get_fallback_data(node_name, fallback_error_content)

    # --- 状態更新、サマリメッセージ生成 ---
    new_state = state.copy()
    # feedbackはリストだが、ここでは計画を一つ入れる想定
    new_state["feedback"] = [feedback_plan]
    current_messages = list(messages)

    if response_message and response_message.content and str(response_message.content).strip():
        logger.info(f"Adding LLM response from {node_name} to messages.")
        current_messages.append(response_message)
    elif response_message:
        logger.warning(f"Skipping adding empty/whitespace LLM response from {node_name} to messages.")

    summary_message_content = f"""製品開発プロセスが完了しました！
最終ステップとして、フィードバック収集計画が作成されました。"""
    if "error_info" in feedback_plan:
        error_detail = feedback_plan.get("error_info", "不明なエラー")
        summary_message_content += f"\n\n*注意: 最終ステップのフィードバック収集計画作成で問題が発生しました ({error_detail[:150]}...)。結果は不完全な可能性があります。"
    else:
        methods = feedback_plan.get('feedback_methods', [])
        stakeholders = feedback_plan.get('target_stakeholders', [])
        summary_message_content += f"\n主な収集方法: {', '.join(methods) if methods else '未定'}"
        summary_message_content += f"\n対象者: {', '.join(stakeholders) if stakeholders else '未定'}"

    summary_message_content += "\n\nこれまでの全ステップの結果は最終状態として保存されています。"
    summary_message = AIMessage(content=summary_message_content)
    current_messages.append(summary_message)

    new_state["messages"] = current_messages
    new_state["current_step"] = node_name # ここを "feedback" に設定
    logger.info(f"Updated state in {node_name}. current_step set to '{node_name}'. Process finished.")
    return new_state


# --- ルーター関数 (変更なし、ログ微修正) ---
def router(state: ProductDevelopmentState) -> str:
    logger.info(f"\n--- Executing Router ---")
    current_step = state.get("current_step")
    logger.info(f"Router received current_step: {current_step}")

    # current_stepがNoneの場合は通常、最初のノード実行前なので、ここでは警告を出さずに次のステップへ
    # ただし、予期せずNoneになった場合は問題
    if current_step is None:
         logger.warning("Router Warning: current_step is None unexpectedly. This might indicate an issue. Defaulting to END.")
         return END # または最初のステップを指定するか、エラー処理

    # ステップ間の遷移を定義
    step_flow = {
        "generate_idea": "step_market_research",
        "market_research": "step_define_product_spec",
        "product_spec": "step_plan_prototype",
        "prototype_plan": "step_collect_feedback",
        "feedback": END # 最後のステップ
    }

    next_node = step_flow.get(current_step)

    if next_node is None:
        logger.error(f"Router Error: No defined next step for current_step '{current_step}'. Ending graph.")
        return END
    elif next_node == END:
        logger.info(f"Router determined next node: END")
    else:
        logger.info(f"Router determined next node: {next_node}")

    return next_node

# --- グラフ構築関数 (変更なし、ログ微修正) ---
def build_product_development_graph() -> StateGraph:
    logger.info("--- Building Product Development Graph ---")
    workflow = StateGraph(ProductDevelopmentState)
    logger.info("Adding nodes...")
    workflow.add_node("step_generate_idea", generate_idea)
    workflow.add_node("step_market_research", market_research)
    workflow.add_node("step_define_product_spec", define_product_spec)
    workflow.add_node("step_plan_prototype", plan_prototype)
    workflow.add_node("step_collect_feedback", collect_feedback)
    logger.info("Nodes added: step_generate_idea, step_market_research, step_define_product_spec, step_plan_prototype, step_collect_feedback")

    logger.info("Setting entry point to 'step_generate_idea'...")
    workflow.set_entry_point("step_generate_idea")
    logger.info("Entry point set.")

    logger.info("Adding conditional edges using the router...")
    # 各ノードの後にルーターを呼び出し、次のノードまたはENDに遷移させる
    workflow.add_conditional_edges(
        "step_generate_idea",
        router,
        {
            "step_market_research": "step_market_research",
            END: END
        }
    )
    workflow.add_conditional_edges(
        "step_market_research",
        router,
        {
            "step_define_product_spec": "step_define_product_spec",
            END: END
        }
    )
    workflow.add_conditional_edges(
        "step_define_product_spec",
        router,
        {
            "step_plan_prototype": "step_plan_prototype",
            END: END
        }
    )
    workflow.add_conditional_edges(
        "step_plan_prototype",
        router,
        {
            "step_collect_feedback": "step_collect_feedback",
            END: END
        }
    )
    # 最後のノードからもルーターを呼び出し、ENDに遷移させる
    workflow.add_conditional_edges(
        "step_collect_feedback",
        router,
        {
            END: END
            # ここで他のパスを追加することも可能（例：フィードバックに基づきループ）
        }
    )
    logger.info("Conditional edges added.")

    logger.info("Compiling graph...")
    compiled_graph = workflow.compile()
    logger.info("Graph compiled successfully.")
    return compiled_graph

# --- ワークフロー実行関数 (変更なし、ログ微修正) ---
def run_product_development(query: str, context: Optional[dict] = None):
    graph = build_product_development_graph()
    initial_state = ProductDevelopmentState( # TypedDictを使って初期化
        messages=[HumanMessage(content=query)],
        current_step=None, # 開始時はNone、エントリーポイントで最初のノードが呼ばれる
        product_idea={},
        market_research={},
        product_spec={},
        prototype_plan={},
        feedback=[]
    )
    if context:
        logger.info(f"Applying initial context: {list(context.keys())}")
        initial_state.update(context) # type: ignore # TypedDictへのupdateは型チェッカーが警告を出すことがある

    logger.info("--- Starting Product Development Workflow ---")
    # ストリーミングではなく、最終結果を一度に取得
    final_state = graph.invoke(initial_state)
    logger.info("\n--- Product Development Workflow Finished ---")
    return final_state

# === メインの実行部分 (変更なし、ログ微修正) ===
if __name__ == "__main__":
    initial_query = "高齢者向けのスマートホームデバイスを開発したいと考えています。アイデアを出してください。"
    logger.info(f"Initial Query: {initial_query}")

    final_state = run_product_development(initial_query)

    print("\n" + "="*30 + " Final State " + "="*30)
    # messagesを除いて表示
    final_state_display = {k: v for k, v in final_state.items() if k != 'messages'}
    pprint.pprint(final_state_display, indent=2, width=120) # 幅を広げる
    # エラーが発生したステップがないか確認
    errors_found = []
    for step in ["product_idea", "market_research", "product_spec", "prototype_plan", "feedback"]:
        step_data = final_state.get(step)
        # feedbackはリストなので最初の要素をチェック
        if step == "feedback" and isinstance(step_data, list) and step_data:
            step_data = step_data[0]

        if isinstance(step_data, dict) and "error_info" in step_data:
             errors_found.append(f"{step}: {step_data['error_info']}")
    if errors_found:
        print("\n--- Errors Encountered During Workflow ---")
        for error in errors_found:
            print(f"- {error}")
    print("="*73)


    print("\n" + "="*30 + " Conversation History (excluding System Messages) " + "="*30)
    if final_state.get("messages"):
        for message in final_state["messages"]:
            role = "Unknown"
            content_to_display = getattr(message, 'content', '')
            if isinstance(message, AIMessage): role = "AI"
            elif isinstance(message, HumanMessage): role = "Human"
            elif isinstance(message, SystemMessage): continue # システムメッセージは表示しない

            content_str = str(content_to_display).strip()
            if content_str:
                print(f"\n--- {role} ---")
                # JSONコンテンツの場合は整形して表示試行
                try:
                    # ```json ``` が含まれるか、全体が{ or [ で始まるかチェック
                    if content_str.startswith('```json') or (content_str.startswith('{') and content_str.endswith('}')) or (content_str.startswith('[') and content_str.endswith(']')):
                        # ```json ``` を除去
                        cleaned_content = re.sub(r'^```(?:json)?\s*\n', '', content_str)
                        cleaned_content = re.sub(r'\n```$', '', cleaned_content)
                        parsed_json = json.loads(cleaned_content.strip())
                        print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
                    else:
                        print(content_str) # JSONでない場合はそのまま表示
                except (json.JSONDecodeError, TypeError):
                     print(content_str) # パース失敗時もそのまま表示
            else:
                 print(f"--- {role} --- (Empty Content)")
    else:
        print("No messages found in the final state.")
    print("="*80)

    # 最後のAIメッセージをサマリとして表示（存在すれば）
    last_message = final_state["messages"][-1] if final_state.get("messages") else None
    if isinstance(last_message, AIMessage) and "製品開発プロセスが完了しました" in str(getattr(last_message, 'content', '')):
        print("\n--- Final Summary from AI ---")
        print(str(getattr(last_message, 'content', '')))

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! generation_config is not default parameter.
                generation_config was transferred to model_kwargs.
                Please confirm that generation_config is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
ERROR:__main__:Router Error: No defined next step for current_step 'define_product_spec'. Ending graph.



============================== Final State ==============================
{ 'current_step': 'define_product_spec',
  'feedback': [],
  'market_research': { 'competitive_advantage': [ '非接触での健康モニタリングによるプライバシー保護',
                                                  'AIによる異常検知と自動通報機能',
                                                  'ミラーとしての自然な使用感',
                                                  '家族とのビデオ通話機能によるコミュニケーション支援'],
                       'competitors': [ { 'name': 'セコム/ALSOKの見守りサービス',
                                          'strengths': 'ブランド力、実績、全国的なサポート体制',
                                          'weaknesses': '月額費用が高い、プライバシーへの懸念'},
                                        { 'name': 'スマートウォッチ型見守りデバイス',
                                          'strengths': '携帯性、緊急通報機能',
                                          'weaknesses': '充電が必要、装着を嫌がる高齢者もいる'},
                                        { 'name': 'センサー付き住宅設備 (人感センサー、開閉センサー)',
                                          'st

# 演習５：プログラミング教育アシスタント

　プログラミング学習者向けのAIアシスタントを開発してください。このアシスタントは以下の機能を持つ必要があります。

- コードの説明と解説
- エラーの診断と修正提案
- プログラミングの概念説明
- 段階的な課題解決のガイダンス
- 学習進捗の追跡と次のステップの提案

In [ ]:
# ライブラリインポート
import os
from typing import List, Dict, Any, TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai.types as safety_types
from google.generativeai.types.generation_types import BlockedPromptException, StopCandidateException
from google.api_core import exceptions as google_api_exceptions
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type, RetryCallState
import json
import re
import pprint
import logging
import uuid
from datetime import datetime # datetime をインポート

# --- (ロギング設定、LLM初期化、invoke_llm_with_retry は変更なし) ---
# ロギング設定
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [%(funcName)s] %(message)s')
logger = logging.getLogger(__name__)

# --- 環境変数設定 (APIキー) ---
# GOOGLE_API_KEY = "YOUR_API_KEY" # 必要に応じて設定
# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# if not os.getenv("GOOGLE_API_KEY"):
#     logger.warning("GOOGLE_API_KEY environment variable not set.")

# --- LLMの初期化 ---
llm_general = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    safety_settings={ # 少し緩和
        safety_types.HarmCategory.HARM_CATEGORY_HARASSMENT: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        safety_types.HarmCategory.HARM_CATEGORY_HATE_SPEECH: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        safety_types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: safety_types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
        safety_types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    },
    temperature=0.5
)

llm_json = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    safety_settings={ # 同上
        safety_types.HarmCategory.HARM_CATEGORY_HARASSMENT: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        safety_types.HarmCategory.HARM_CATEGORY_HATE_SPEECH: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        safety_types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: safety_types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
        safety_types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: safety_types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    },
    generation_config={"response_mime_type": "application/json"},
    temperature=0.1
)

# --- リトライ付きLLM呼び出し関数 (変更なし) ---
class EmptyResponseError(Exception): pass
RETRYABLE_EXCEPTIONS = (google_api_exceptions.ResourceExhausted, google_api_exceptions.ServiceUnavailable, google_api_exceptions.DeadlineExceeded, google_api_exceptions.Aborted, google_api_exceptions.InternalServerError, EmptyResponseError, BlockedPromptException)
def log_retry_attempt(retry_state: RetryCallState): logger.warning(f"LLM call failed (attempt {retry_state.attempt_number}). Retrying in {retry_state.next_action.sleep} seconds. Error: {type(retry_state.outcome.exception()).__name__}: {str(retry_state.outcome.exception())[:200]}")
@retry(stop=stop_after_attempt(3), wait=wait_fixed(2), retry=retry_if_exception_type(RETRYABLE_EXCEPTIONS), after=log_retry_attempt)
def invoke_llm_with_retry(llm_instance: ChatGoogleGenerativeAI, messages_input: List[BaseMessage], caller: str = "unknown") -> AIMessage:
    attempt_num = getattr(invoke_llm_with_retry.retry.statistics, 'attempt_number', 1); logger.info(f"Attempting LLM call from {caller} (attempt {attempt_num})...")
    try: response = llm_instance.invoke(messages_input)
    except (BlockedPromptException, StopCandidateException) as safety_exception: logger.error(f"LLM call blocked by API from {caller} (attempt {attempt_num}): {safety_exception}"); raise
    except google_api_exceptions.ApiException as api_exc: logger.error(f"Google API error during LLM call from {caller} (attempt {attempt_num}): {type(api_exc).__name__} - {api_exc}"); raise
    except Exception as e: logger.error(f"Unexpected error during LLM invocation from {caller} (attempt {attempt_num}): {type(e).__name__} - {e}", exc_info=True); raise
    if not response.content or not str(response.content).strip(): logger.warning(f"LLM from {caller} returned empty content unexpectedly on attempt {attempt_num}, raising error for retry..."); raise EmptyResponseError("LLM returned empty or whitespace content.")
    finish_reason = response.response_metadata.get("finish_reason");
    if finish_reason == "SAFETY": logger.critical(f"LLM call from {caller} stopped due to SAFETY unexpectedly (attempt {attempt_num}) despite settings."); raise Exception("LLM response blocked by safety filter unexpectedly.")
    logger.info(f"LLM call from {caller} attempt {attempt_num} successful."); return response

# --- 状態定義 (変更なし) ---
class ProgrammingAssistantState(TypedDict):
    messages: List[BaseMessage]
    current_task: Optional[str]
    context_data: Dict[str, Any]
    learner_profile: Dict[str, Any]
    progress_log: List[Dict[str, Any]]
    last_ai_response: Optional[str]

# --- ユーティリティ関数 (変更なし) ---
def extract_code(text: str) -> Optional[str]: match = re.search(r'```(?:\w*\n)?(.*)```', text, re.DOTALL); return match.group(1).strip() if match else None
def get_last_user_message(messages: List[BaseMessage]) -> Optional[str]:
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage): return str(msg.content)
    return None

# --- ノード関数 ---

# 1. リクエスト分類ルーター (変更なし)
def route_request(state: ProgrammingAssistantState) -> Dict[str, Any]:
    logger.info("Executing request router..."); last_user_msg = get_last_user_message(state['messages'])
    if not last_user_msg: logger.warning("No user message found for routing."); return {"current_task": "unknown", "context_data": {}}
    system_prompt = SystemMessage(content="""あなたはプログラミング教育アシスタントのリクエスト分類器です。ユーザーの最新のメッセージを分析し、その主な意図を判断してください。
以下のカテゴリから最も適切なものを**1つだけ**選択し、関連情報（コード、エラーメッセージ、概念名、問題記述）を抽出してください。
カテゴリ: explain_code, debug_error, explain_concept, solve_problem, ask_progress, unknown
関連情報が見つからない場合は、該当するキーの値を null または空文字列にしてください。
出力は必ず以下のJSON形式にしてください:
{ "task": "選択されたカテゴリ名", "context": { "code": "...", "error_message": "...", "concept_name": "...", "problem_description": "...", "original_query": "..." } }""")
    input_messages = [system_prompt, HumanMessage(content=last_user_msg)]; raw_response_content = ""
    try:
        response = invoke_llm_with_retry(llm_json, input_messages, caller="route_request"); raw_response_content = response.content
        json_str_to_parse = raw_response_content; json_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', raw_response_content, re.DOTALL | re.IGNORECASE)
        if json_match: json_str_to_parse = json_match.group(1).strip(); logger.info("Found and removed ```json ``` block from router response.")
        else: json_str_to_parse = raw_response_content.strip(); logger.info("No ```json ``` block found in router response, parsing as is.")
        if not json_str_to_parse: logger.error("Router LLM response content is empty after stripping markdown."); raise json.JSONDecodeError("Empty content after stripping markdown", "", 0)
        response_json = json.loads(json_str_to_parse); task = response_json.get("task", "unknown"); context = response_json.get("context", {})
        context['original_query'] = last_user_msg
        if task == "explain_code" and not context.get("code"): context["code"] = extract_code(last_user_msg) or last_user_msg
        if task == "debug_error" and not context.get("code"): context["code"] = extract_code(last_user_msg)
        if task == "debug_error" and not context.get("error_message"): lines = last_user_msg.splitlines(); error_lines = [line for line in lines if "error" in line.lower() or "exception" in line.lower() or line.strip().startswith("Traceback")]; context["error_message"] = "\n".join(error_lines) if error_lines else None
        logger.info(f"Routing determined task: {task}, Context keys: {list(context.keys())}"); return {"current_task": task, "context_data": context}
    except json.JSONDecodeError as e: logger.error(f"Failed to decode JSON response from router LLM: {e}. Response content: {raw_response_content}"); return {"current_task": "unknown", "context_data": {"original_query": last_user_msg}}
    except Exception as e: logger.error(f"Error during request routing: {e}", exc_info=True); return {"current_task": "unknown", "context_data": {"original_query": last_user_msg}}

# 2. 各タスク実行ノード (変更なし)
def execute_task(state: ProgrammingAssistantState, task_name: str, system_message_content: str, llm_instance: ChatGoogleGenerativeAI) -> ProgrammingAssistantState:
    logger.info(f"Executing task: {task_name}..."); messages = state['messages']; context_data = state.get('context_data', {})
    try: formatted_prompt = system_message_content.format(**context_data)
    except KeyError as e: logger.warning(f"Missing key '{e}' in context_data for task {task_name}. Using raw prompt."); formatted_prompt = system_message_content
    system_prompt = SystemMessage(content=formatted_prompt); input_messages = [system_prompt] + messages; new_state = state.copy()
    try:
        response = invoke_llm_with_retry(llm_instance, input_messages, caller=task_name); ai_response_content = response.content
        ai_message = AIMessage(content=ai_response_content); new_state['messages'] = messages + [ai_message]; new_state['last_ai_response'] = ai_response_content
        logger.info(f"Task {task_name} executed successfully.")
    except Exception as e:
        logger.error(f"Error executing task {task_name}: {e}", exc_info=True)
        error_message_content = f"Sorry, I encountered an error trying to {task_name.replace('_', ' ')}. Please try again or rephrase your request."
        error_message = AIMessage(content=error_message_content); new_state['messages'] = messages + [error_message]; new_state['last_ai_response'] = error_message_content
    return new_state

def explain_code_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    prompt = "あなたは親切で忍耐強いプログラミングチューターです。ユーザーが提供した以下のコードスニペットについて、明確かつ簡潔に説明してください...\n提供されたコード:\n```\n{code}\n```"; code = state.get('context_data', {}).get('code')
    if not code: logger.warning("No code found..."); ai_message = AIMessage(content="コードの説明をご希望ですね..."); new_state = state.copy(); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_message.content; return new_state
    else: return execute_task(state, "explain_code", prompt, llm_general)

def debug_error_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    prompt = "あなたは熟練したデバッガーで、プログラミング学習者を支援しています。以下のコードとエラーメッセージを分析してください...\nコード:\n```\n{code}\n```\nエラーメッセージ:\n```\n{error_message}\n```"; code = state.get('context_data', {}).get('code'); error_msg = state.get('context_data', {}).get('error_message')
    if not code or not error_msg: logger.warning(f"Missing code or error... Code: {bool(code)}, Error: {bool(error_msg)}"); missing = ([] if code else ["エラーが発生したコード"]) + ([] if error_msg else ["表示されたエラーメッセージ"]); ai_message = AIMessage(content=f"エラーのデバッグをお手伝いします。{'と'.join(missing)}を提示してください。"); new_state = state.copy(); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_message.content; return new_state
    else: return execute_task(state, "debug_error", prompt, llm_general)

def explain_concept_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    prompt = "あなたは明確で簡潔なプログラミングインストラクターです。以下のプログラミング概念を初心者に説明してください: '{concept_name}'...\n説明する概念: {concept_name}"; concept = state.get('context_data', {}).get('concept_name')
    if not concept: logger.warning("No concept name found..."); original_query = state.get('context_data', {}).get('original_query', "your request"); ai_message = AIMessage(content=f"プログラミングの概念について説明しますね。どの概念について知りたいですか？ 例:「'{original_query}' について説明して」"); new_state = state.copy(); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_message.content; return new_state
    else: return execute_task(state, "explain_concept", prompt, llm_general)

def guide_problem_solving_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    prompt = "あなたはソクラテス式問答法のプログラミングチューターです。ユーザーはこの問題を解決しようとしています: '{problem_description}'...\n解決したい問題: {problem_description}"; problem = state.get('context_data', {}).get('problem_description')
    if not problem: logger.warning("No problem description found..."); ai_message = AIMessage(content="問題解決のお手伝いをします。どのような問題に取り組んでいますか？"); new_state = state.copy(); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_message.content; return new_state
    else: return execute_task(state, "solve_problem", prompt, llm_general)

def handle_unknown_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    logger.info("Handling unknown request type."); last_user_msg = state.get('context_data', {}).get('original_query', 'your request')
    ai_message = AIMessage(content=f"すみません、'{last_user_msg[:50]}...' について、どのように対応すればよいか判断できませんでした..."); new_state = state.copy(); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_message.content; new_state['current_task'] = 'unknown'; return new_state

def ask_progress_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    logger.info("Handling ask_progress request."); learner_profile = state.get('learner_profile', {}); topics_covered = learner_profile.get('topics_covered', []); skill_level = learner_profile.get('skill_level', 'Beginner')
    progress_summary = f"現在のスキルレベル: {skill_level}\n" + (f"これまでに学習したトピック: {', '.join(topics_covered)}\n" if topics_covered else "まだ学習記録がありません。\n")
    prompt = f"あなたはプログラミング学習のアドバイザーです。学習者の現在の状況は以下の通りです。\n{progress_summary}\nこれに基づき、次に学習すると良いと思われるステップやトピックを1～2つ提案してください..."; system_prompt = SystemMessage(content=prompt); input_messages = [system_prompt]; new_state = state.copy()
    try:
        response = invoke_llm_with_retry(llm_general, input_messages, caller="ask_progress_suggestion"); suggestion = response.content
        ai_response_content = progress_summary + "\n" + suggestion; ai_message = AIMessage(content=ai_response_content); new_state['messages'] = state['messages'] + [ai_message]; new_state['last_ai_response'] = ai_response_content
        logger.info("Progress summary and next step suggestion provided.")
    except Exception as e:
        logger.error(f"Error generating next step suggestion: {e}", exc_info=True); error_message_content = progress_summary + "\n申し訳ありません。次のステップの提案中にエラーが発生しました。"
        error_message = AIMessage(content=error_message_content); new_state['messages'] = state['messages'] + [error_message]; new_state['last_ai_response'] = error_message_content
    new_state['current_task'] = 'ask_progress'; return new_state


# 3. 進捗更新・次のステップ提案ノード (修正: last_ai_response を上書きしない)
def update_progress_node(state: ProgrammingAssistantState) -> ProgrammingAssistantState:
    """タスク完了後、学習者のプロファイルとログを更新し、次の行動を促すメッセージを必要に応じて追加"""
    logger.info("Executing update_progress node...")
    current_task = state.get('current_task')
    context_data = state.get('context_data', {})
    learner_profile = state.get('learner_profile', {'topics_covered': [], 'skill_level': 'Beginner'})
    progress_log = state.get('progress_log', [])
    # last_ai_response は前のノードから引き継いだものをそのまま使う

    learned_topic = None
    if current_task == 'explain_concept':
        learned_topic = context_data.get('concept_name')
    elif current_task == 'solve_problem':
        learned_topic = f"Problem solving ({context_data.get('problem_description', 'N/A')[:30]}...)"
    # 他のタスクタイプは直接的な学習とはみなしにくいので、ここではプロファイル更新対象外

    # 学習プロファイル更新
    if learned_topic and learned_topic not in learner_profile['topics_covered']:
        logger.info(f"Adding '{learned_topic}' to topics covered.")
        learner_profile['topics_covered'].append(learned_topic)

    # 進捗ログ追加
    log_entry = {
        'id': str(uuid.uuid4()),
        'task': current_task,
        'context': {k: v for k, v in context_data.items() if k != 'original_query'},
        'timestamp': datetime.now().isoformat()
    }
    progress_log.append(log_entry)
    logger.info(f"Added log entry for task: {current_task}")

    new_state = state.copy()
    new_state['learner_profile'] = learner_profile
    new_state['progress_log'] = progress_log

    # --- ▼▼▼ 修正点 ▼▼▼ ---
    # 次の行動を促すメッセージを追加するが、last_ai_response は変更しない
    # ask_progress の場合は既に次の提案が含まれているので、このメッセージは不要
    if current_task != 'ask_progress':
        next_prompt_msg_content = "他に何かお手伝いできることはありますか？ コードの説明、エラーの修正、概念の説明、問題解決のガイドなど、お気軽にお尋ねください。"
        next_prompt_msg = AIMessage(content=next_prompt_msg_content)
        # メッセージリストには追加する
        new_state['messages'] = new_state['messages'] + [next_prompt_msg]
        # last_ai_response は上書きしない！前のノード（例: explain_concept_node）の応答が維持される
        # new_state['last_ai_response'] = next_prompt_msg.content # この行を削除またはコメントアウト
        logger.info("Appended 'Anything else?' prompt message.")
    # --- ▲▲▲ 修正点 ▲▲▲ ---

    # 次のターンに備えてリセット
    new_state['current_task'] = None
    new_state['context_data'] = {}
    logger.info("Progress updated and state reset for next turn.")

    return new_state


# --- グラフ構築 (変更なし) ---
def build_assistant_graph() -> StateGraph:
    logger.info("Building the programming assistant graph...")
    workflow = StateGraph(ProgrammingAssistantState)
    workflow.add_node("route_request", route_request)
    workflow.add_node("explain_code", explain_code_node)
    workflow.add_node("debug_error", debug_error_node)
    workflow.add_node("explain_concept", explain_concept_node)
    workflow.add_node("solve_problem", guide_problem_solving_node)
    workflow.add_node("ask_progress", ask_progress_node)
    workflow.add_node("handle_unknown", handle_unknown_node)
    workflow.add_node("update_progress", update_progress_node)
    workflow.set_entry_point("route_request")
    workflow.add_conditional_edges("route_request", lambda x: x["current_task"], { "explain_code": "explain_code", "debug_error": "debug_error", "explain_concept": "explain_concept", "solve_problem": "solve_problem", "ask_progress": "ask_progress", "unknown": "handle_unknown", })
    workflow.add_edge("explain_code", "update_progress")
    workflow.add_edge("debug_error", "update_progress")
    workflow.add_edge("explain_concept", "update_progress")
    workflow.add_edge("solve_problem", "update_progress")
    workflow.add_edge("ask_progress", "update_progress")
    workflow.add_edge("handle_unknown", "update_progress")
    workflow.add_edge("update_progress", END)
    logger.info("Graph nodes and edges configured.")
    compiled_graph = workflow.compile(); logger.info("Graph compiled successfully.")
    return compiled_graph

# --- メイン実行部分 (変更なし) ---
if __name__ == "__main__":
    print("プログラミング教育アシスタントへようこそ！")
    print("終了するには 'quit' または 'exit' と入力してください。")
    assistant_app = build_assistant_graph()
    initial_state = ProgrammingAssistantState(messages=[], current_task=None, context_data={}, learner_profile={'topics_covered': [], 'skill_level': 'Beginner'}, progress_log=[], last_ai_response=None)
    current_state = initial_state.copy()
    while True:
        user_input = input("\nあなた: ")
        if user_input.lower() in ["quit", "exit"]: print("アシスタントを終了します。お疲れ様でした！"); break
        if not user_input.strip(): continue
        current_state['messages'] = current_state['messages'] + [HumanMessage(content=user_input)]
        current_state['current_task'] = None; current_state['context_data'] = {}; current_state['last_ai_response'] = None
        print("アシスタントが考え中...")
        try:
            invocation_input = {"messages": current_state['messages'], "learner_profile": current_state['learner_profile'], "progress_log": current_state['progress_log']}
            # logger.debug(f"Invoking graph with input keys: {list(invocation_input.keys())}") # デバッグ用
            final_turn_state = assistant_app.invoke(invocation_input)
            # logger.debug(f"Graph invocation finished. Final state keys: {list(final_turn_state.keys())}") # デバッグ用
            current_state = final_turn_state
            # last_ai_response にタスク実行ノードの応答が入っているはず
            if current_state.get('last_ai_response'):
                 print("\nAI:", current_state['last_ai_response'])
            else:
                 # フォールバック：もし last_ai_response がない場合（エラーなど）
                 last_msg = current_state['messages'][-2] if len(current_state.get('messages', [])) >= 2 and isinstance(current_state['messages'][-1], AIMessage) and "他に何かお手伝い" in current_state['messages'][-1].content else current_state['messages'][-1] if current_state.get('messages') else None
                 if isinstance(last_msg, AIMessage):
                     print("\nAI:", last_msg.content)
                 else:
                     logger.warning("Could not determine the primary AI response to display.")
                     print("\nAI: (応答を特定できませんでした)")
        except Exception as e:
            logger.error(f"An error occurred during graph invocation: {e}", exc_info=True)
            print("\nAI: 申し訳ありません、予期せぬエラーが発生しました。もう一度試してみてください。")
            current_state['current_task'] = None; current_state['context_data'] = {}; current_state['last_ai_response'] = "申し訳ありません、予期せぬエラーが発生しました。もう一度試してみてください。"

    print("\n" + "="*30 + " Final Learner State " + "="*30)
    pprint.pprint({"Learner Profile": current_state.get('learner_profile'), "Progress Log Count": len(current_state.get('progress_log', []))})
    print("="*80)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! generation_config is not default parameter.
                generation_config was transferred to model_kwargs.
                Please confirm that generation_config is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


プログラミング教育アシスタントへようこそ！
終了するには 'quit' または 'exit' と入力してください。

あなた: pythonでhello world教えて。
アシスタントが考え中...

AI: もちろんです！Pythonで「Hello, World!」と表示する最も簡単なコードは以下の通りです。

```python
print("Hello, World!")
```

**コードの説明:**

*   **`print()`**: これはPythonの組み込み関数で、指定された内容を画面（コンソール）に出力するために使われます。
*   **`"Hello, World!"`**: これは文字列リテラルです。`print()`関数に渡すことで、この文字列が出力されます。 文字列は、ダブルクォーテーション (`"`) またはシングルクォーテーション (`'`) で囲みます。

**このコードをどうやって実行するか:**

1.  **Pythonがインストールされているか確認:** まず、あなたのコンピュータにPythonがインストールされていることを確認してください。もしインストールされていなければ、[Pythonの公式サイト](https://www.python.org/) からダウンロードしてインストールしてください。
2.  **コードをファイルに保存:** テキストエディタ（メモ帳など）を開き、上記のコードを入力します。そして、`hello.py` のような名前でファイルを保存します。`.py` はPythonファイルの拡張子です。
3.  **コマンドプロンプト/ターミナルで実行:** コマンドプロンプト（Windowsの場合）またはターミナル（macOS/Linuxの場合）を開き、保存したファイルがあるディレクトリに移動します。そして、以下のコマンドを実行します。

    ```bash
    python hello.py
    ```

    すると、画面に `Hello, World!` と表示されるはずです。

**補足:**

*   Pythonは大文字と小文字を区別します。`Print()` や `PRINT()` と書くとエラーになります。
*   `print()` 関数は、文字列だけでなく、数値や変数な

## まとめ

このノートブックでは、LangChainエコシステムを活用した4つの実践的な演習問題に取り組みました。

1. **インテリジェントカスタマーサポートボット**：商品情報にアクセスし、顧客の質問に答えるボットを開発しました。
2. **ドキュメント分析エージェント**：契約書を分析し、重要な条項を抽出するエージェントを構築しました。
3. **研究アシスタントエージェント**：複数のツールを統合し、研究者の学術活動をサポートするエージェントを開発しました。
4. **マルチステップ意思決定エージェント**：LangGraphを使用して、製品開発プロセスをサポートするステートフルなワークフローを構築しました。
5. **プログラミング教育アシスタント**：学習者の習熟度や質問内容を分析し、ヒントの提示やコード解説、学習進捗の管理をパーソナライズして提供する教育支援エージェントを構築しました。

これらの演習を通じて、LangChainの基本コンポーネント、LangGraphによるワークフロー管理、そして実践的なアーキテクチャ設計について学びました。これらの知識を活用して、さらに複雑で高度なAIエージェントを開発することができます。